In [1]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# Cell 0: setup and frozen observation protocol
#
# Scientific question
# ---------------------------------------------------------------
# Can a SINGLE continuous multivariate time series be sliced
# into multi-resolution endpoint families such that the
# extrapolated persistent generator recovers the instantaneous
# network active at each slice start?
#
# Physical system:
#   EXACTLY the frozen Stage-4 N=8 benchmark.
#
# What changes:
#   observation protocol only.
#
# Old:
#   many independent initial conditions
#
# New:
#   one continuous trajectory
#   -> many temporally indexed slice starts
#   -> four endpoint horizons per start
# ================================================================

from pathlib import Path
from itertools import combinations, combinations_with_replacement
from collections import Counter, defaultdict

import importlib
import numpy as np
import pandas as pd
import sympy as sp


# ================================================================
# Frozen physical-system reproducibility
# ================================================================

SEED = 20260811

weight_rng = np.random.default_rng(
    SEED
)

trajectory_rng = np.random.default_rng(
    SEED + 202
)


# ================================================================
# Output directory
# ================================================================

OUTPUT_DIR = Path(
    "./stage6_n8_timeseries_slicing"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# Physical-system dimensions
# ================================================================

N = 8

nodes = tuple(
    range(1, N + 1)
)

x_symbols = sp.symbols(
    f"x1:{N + 1}"
)


# ================================================================
# Frozen nonlinear interaction
#
# phi(x) = x + 0.5 x^2
# ================================================================

LAMBDA = 0.5
LAMBDA_EXACT = sp.Rational(1, 2)


# ================================================================
# ONE-TIME-SERIES observation protocol
#
# Fixed physical snapshot duration:
#
#       tau_snapshot = 0.06
#
# Dense trajectory observation:
#
#       dt_obs = 0.0005
#
# Multi-resolution slice endpoints:
#
#       eps = [0.005, 0.0075, 0.010, 0.015]
#
# All eps values are exact integer multiples of dt_obs.
# ================================================================

TAU_SNAPSHOT = 0.060

DT_OBS = 0.0005

EPS_SLICE = np.array(
    [
        0.0050,
        0.0075,
        0.0100,
        0.0150,
    ],
    dtype=float,
)

EPS_MAX = float(
    EPS_SLICE.max()
)


# ================================================================
# Repeated temporal protocol
#
# One continuous trajectory:
#
#   G1 -> G2 -> ... -> G6
#      -> G1 -> G2 -> ...
#
# Six cycles are generated initially.
#
# Block roles:
#
#   cycles 0-3 : fitting
#   cycle 4    : validation
#   cycle 5    : external test
# ================================================================

N_CYCLES = 6

FIT_CYCLES = (
    0, 1, 2, 3
)

VALIDATION_CYCLES = (
    4,
)

TEST_CYCLES = (
    5,
)


# ================================================================
# Slice-start spacing
#
# This controls how densely starting states are extracted from
# the ONE trajectory.
#
# It does NOT define independent experiments.
# ================================================================

SLICE_STRIDE = 0.0015


# ================================================================
# Exact grid compatibility
# ================================================================

def integer_steps(
    duration,
    dt=DT_OBS,
):
    steps = int(
        round(
            float(duration)
            /
            float(dt)
        )
    )

    if not np.isclose(
        steps * dt,
        duration,
        rtol=0.0,
        atol=1e-14,
    ):
        raise ValueError(
            f"{duration} is not an integer "
            f"multiple of dt={dt}."
        )

    return steps


SNAPSHOT_STEPS = integer_steps(
    TAU_SNAPSHOT
)

EPS_STEPS = np.array(
    [
        integer_steps(epsilon)
        for epsilon
        in EPS_SLICE
    ],
    dtype=int,
)

SLICE_STRIDE_STEPS = integer_steps(
    SLICE_STRIDE
)

CYCLE_STEPS = (
    6
    *
    SNAPSHOT_STEPS
)

TOTAL_STEPS = (
    N_CYCLES
    *
    CYCLE_STEPS
)

TOTAL_TIME = (
    TOTAL_STEPS
    *
    DT_OBS
)


# ================================================================
# Load frozen TSC v3.6 core
# ================================================================

try:
    import TSC_AGLASSO_v3_6 as tsc_core

except ModuleNotFoundError:
    import TSC_AGLASSO as tsc_core


tsc_core = importlib.reload(
    tsc_core
)

assert (
    getattr(
        tsc_core,
        "_IMPLEMENTATION_VERSION",
        None,
    )
    ==
    "v3.6"
)


# ================================================================
# Basic configuration audit
# ================================================================

print("=" * 70)
print("N=8 ONE-TIME-SERIES SLICING EXPERIMENT")
print("=" * 70)

print(
    f"N                           : "
    f"{N}"
)

print(
    f"snapshot duration           : "
    f"{TAU_SNAPSHOT}"
)

print(
    f"dense observation dt        : "
    f"{DT_OBS}"
)

print(
    f"steps / snapshot            : "
    f"{SNAPSHOT_STEPS}"
)

print(
    f"slice epsilon values        : "
    f"{EPS_SLICE}"
)

print(
    f"epsilon steps               : "
    f"{EPS_STEPS}"
)

print(
    f"slice-start stride          : "
    f"{SLICE_STRIDE}"
)

print(
    f"slice stride steps          : "
    f"{SLICE_STRIDE_STEPS}"
)

print(
    f"cycles                      : "
    f"{N_CYCLES}"
)

print(
    f"total trajectory duration   : "
    f"{TOTAL_TIME}"
)

print(
    f"fit cycles                  : "
    f"{FIT_CYCLES}"
)

print(
    f"validation cycles           : "
    f"{VALIDATION_CYCLES}"
)

print(
    f"test cycles                 : "
    f"{TEST_CYCLES}"
)

print(
    f"TSC implementation          : "
    f"{tsc_core._IMPLEMENTATION_VERSION}"
)


# ================================================================
# Expected nominal number of eligible interior starts
#
# No start is allowed if its largest epsilon crosses a snapshot
# boundary.
# ================================================================

LAST_LOCAL_START_STEP = (
    SNAPSHOT_STEPS
    -
    int(EPS_STEPS.max())
)

START_OFFSETS = np.arange(
    0,
    LAST_LOCAL_START_STEP + 1,
    SLICE_STRIDE_STEPS,
    dtype=int,
)

STARTS_PER_SNAPSHOT_PER_CYCLE = len(
    START_OFFSETS
)

print()
print("Nominal slicing capacity")
print("-" * 70)

print(
    f"eligible starts / snapshot / cycle : "
    f"{STARTS_PER_SNAPSHOT_PER_CYCLE}"
)

print(
    f"eligible starts / snapshot total   : "
    f"{STARTS_PER_SNAPSHOT_PER_CYCLE * N_CYCLES}"
)

print(
    f"fit starts / snapshot              : "
    f"{STARTS_PER_SNAPSHOT_PER_CYCLE * len(FIT_CYCLES)}"
)

print(
    f"validation starts / snapshot       : "
    f"{STARTS_PER_SNAPSHOT_PER_CYCLE * len(VALIDATION_CYCLES)}"
)

print(
    f"test starts / snapshot             : "
    f"{STARTS_PER_SNAPSHOT_PER_CYCLE * len(TEST_CYCLES)}"
)


# ================================================================
# Hard guards
# ================================================================

assert EPS_MAX < TAU_SNAPSHOT

assert (
    set(FIT_CYCLES)
    .isdisjoint(
        VALIDATION_CYCLES
    )
)

assert (
    set(FIT_CYCLES)
    .isdisjoint(
        TEST_CYCLES
    )
)

assert (
    set(VALIDATION_CYCLES)
    .isdisjoint(
        TEST_CYCLES
    )
)

assert (
    set(FIT_CYCLES)
    |
    set(VALIDATION_CYCLES)
    |
    set(TEST_CYCLES)
) == set(
    range(N_CYCLES)
)


print()
print("=" * 70)
print("Cell 0 PASSED.")
print("=" * 70)

N=8 ONE-TIME-SERIES SLICING EXPERIMENT
N                           : 8
snapshot duration           : 0.06
dense observation dt        : 0.0005
steps / snapshot            : 120
slice epsilon values        : [0.005  0.0075 0.01   0.015 ]
epsilon steps               : [10 15 20 30]
slice-start stride          : 0.0015
slice stride steps          : 3
cycles                      : 6
total trajectory duration   : 2.16
fit cycles                  : (0, 1, 2, 3)
validation cycles           : (4,)
test cycles                 : (5,)
TSC implementation          : v3.6

Nominal slicing capacity
----------------------------------------------------------------------
eligible starts / snapshot / cycle : 31
eligible starts / snapshot total   : 186
fit starts / snapshot              : 124
validation starts / snapshot       : 31
test starts / snapshot             : 31

Cell 0 PASSED.


In [2]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# Cell 1: exact frozen Stage-4 physical system
#
# IMPORTANT
# ---------------------------------------------------------------
# This cell reproduces the physical system from the original
# Stage-4 N=8 benchmark.
#
# Nothing in the microscopic dynamics is changed:
#
#   - same 12 microscopic edges
#   - same heterogeneous edge weights
#   - same six temporal snapshots
#   - same nonlinear pairwise interaction
#   - same two persistent native triads
#
# Only the observation protocol will change in later cells.
# ================================================================


# ================================================================
# Microscopic pairwise graph
# ================================================================

microscopic_edges = (
    (1, 2),
    (2, 3),
    (3, 4),
    (4, 5),
    (5, 6),
    (6, 7),
    (7, 8),
    (8, 1),
    (1, 3),
    (2, 4),
    (3, 5),
    (5, 7),
)

assert len(microscopic_edges) == 12
assert len(set(microscopic_edges)) == 12


# ================================================================
# Frozen heterogeneous edge weights
#
# EXACT Stage-4 construction:
#
#   integer weight ~ Uniform{800,...,1200}
#   physical weight = integer / 1000
#
# weight_rng was initialized in Cell 0 with:
#
#       SEED = 20260811
#
# ================================================================

edge_weight_integer = {
    edge:
        int(
            weight_rng.integers(
                800,
                1201,
            )
        )

    for edge
    in microscopic_edges
}

edge_weight = {
    edge:
        value / 1000.0

    for edge, value
    in edge_weight_integer.items()
}

edge_weight_exact = {
    edge:
        sp.Rational(
            value,
            1000,
        )

    for edge, value
    in edge_weight_integer.items()
}


# ================================================================
# Frozen six-snapshot temporal protocol
#
# Every microscopic edge appears exactly twice over one complete
# six-snapshot cycle.
# ================================================================

snapshots = (

    # G1
    (
        (1, 2),
        (3, 4),
        (5, 6),
        (7, 8),
    ),

    # G2
    (
        (2, 3),
        (4, 5),
        (6, 7),
        (8, 1),
    ),

    # G3
    (
        (1, 3),
        (2, 4),
        (3, 5),
        (5, 7),
    ),

    # G4
    (
        (1, 2),
        (4, 5),
        (7, 8),
        (3, 5),
    ),

    # G5
    (
        (2, 3),
        (5, 6),
        (8, 1),
        (5, 7),
    ),

    # G6
    (
        (3, 4),
        (6, 7),
        (1, 3),
        (2, 4),
    ),
)


# ================================================================
# Snapshot consistency audit
# ================================================================

snapshot_edge_counts = Counter(
    edge

    for snapshot
    in snapshots

    for edge
    in snapshot
)

assert len(snapshots) == 6

assert all(
    len(snapshot) == 4
    for snapshot
    in snapshots
)

assert (
    set(snapshot_edge_counts)
    ==
    set(microscopic_edges)
)

assert all(
    snapshot_edge_counts[edge] == 2
    for edge
    in microscopic_edges
)


# ================================================================
# Nonlinear pairwise interaction
#
#       phi(x) = x + 0.5 x^2
# ================================================================

def phi_symbolic(z):
    return (
        z
        +
        LAMBDA_EXACT * z**2
    )


def phi_numeric(z):
    return (
        z
        +
        LAMBDA * z**2
    )


# ================================================================
# Pairwise microscopic vector field
#
# For edge (i,j):
#
#   F_i = w_ij [phi(x_j) - phi(x_i)]
#   F_j = -F_i
#
# Hence every edge field conserves:
#
#       sum_i x_i
# ================================================================

def edge_field_symbolic(
    edge,
):

    i, j = edge

    if edge not in edge_weight_exact:
        raise KeyError(
            f"Unknown microscopic edge: "
            f"{edge}"
        )

    w = edge_weight_exact[
        edge
    ]

    field = sp.zeros(
        N,
        1,
    )

    interaction = (
        w
        *
        (
            phi_symbolic(
                x_symbols[j - 1]
            )
            -
            phi_symbolic(
                x_symbols[i - 1]
            )
        )
    )

    field[
        i - 1
    ] += interaction

    field[
        j - 1
    ] -= interaction

    return field


def edge_field_numeric(
    X,
    edge,
):

    i, j = edge

    if edge not in edge_weight:
        raise KeyError(
            f"Unknown microscopic edge: "
            f"{edge}"
        )

    w = edge_weight[
        edge
    ]

    F = np.zeros_like(
        X,
        dtype=float,
    )

    interaction = (
        w
        *
        (
            phi_numeric(
                X[..., j - 1]
            )
            -
            phi_numeric(
                X[..., i - 1]
            )
        )
    )

    F[
        ..., i - 1
    ] += interaction

    F[
        ..., j - 1
    ] -= interaction

    return F


# ================================================================
# Persistent native triadic interactions
#
# Frozen Stage-4 definition:
#
# For triad h={i,j,k},
#
#   T_i =
#       g x_j x_k
#       - g/2 x_i x_j
#       - g/2 x_i x_k
#
# with cyclic permutations for j and k.
#
# These interactions are present in EVERY snapshot.
# ================================================================

NATIVE_TRIADS = (
    (1, 2, 3),
    (2, 5, 8),
)

NATIVE_G = 0.02

NATIVE_G_EXACT = sp.Rational(
    1,
    50,
)


def triad_field_symbolic(
    triad,
    g=NATIVE_G_EXACT,
):

    i, j, k = triad

    xi = x_symbols[
        i - 1
    ]

    xj = x_symbols[
        j - 1
    ]

    xk = x_symbols[
        k - 1
    ]

    field = sp.zeros(
        N,
        1,
    )

    field[
        i - 1
    ] += (
        g * xj * xk
        -
        g / 2 * xi * xj
        -
        g / 2 * xi * xk
    )

    field[
        j - 1
    ] += (
        g * xi * xk
        -
        g / 2 * xj * xi
        -
        g / 2 * xj * xk
    )

    field[
        k - 1
    ] += (
        g * xi * xj
        -
        g / 2 * xk * xi
        -
        g / 2 * xk * xj
    )

    return field


def triad_field_numeric(
    X,
    triad,
    g=NATIVE_G,
):

    i, j, k = triad

    xi = X[
        ..., i - 1
    ]

    xj = X[
        ..., j - 1
    ]

    xk = X[
        ..., k - 1
    ]

    F = np.zeros_like(
        X,
        dtype=float,
    )

    F[
        ..., i - 1
    ] += (
        g * xj * xk
        -
        0.5 * g * xi * xj
        -
        0.5 * g * xi * xk
    )

    F[
        ..., j - 1
    ] += (
        g * xi * xk
        -
        0.5 * g * xj * xi
        -
        0.5 * g * xj * xk
    )

    F[
        ..., k - 1
    ] += (
        g * xi * xj
        -
        0.5 * g * xk * xi
        -
        0.5 * g * xk * xj
    )

    return F


# ================================================================
# Precompute symbolic microscopic fields
# ================================================================

edge_fields_symbolic = {
    edge:
        edge_field_symbolic(
            edge
        )

    for edge
    in microscopic_edges
}


native_triad_fields_symbolic = {
    triad:
        triad_field_symbolic(
            triad
        )

    for triad
    in NATIVE_TRIADS
}


# ================================================================
# Persistent native contribution
# ================================================================

native_total_symbolic = sum(
    native_triad_fields_symbolic.values(),
    sp.zeros(
        N,
        1,
    ),
)


def native_total_numeric(
    X,
):

    F = np.zeros_like(
        X,
        dtype=float,
    )

    for triad in NATIVE_TRIADS:

        F += triad_field_numeric(
            X,
            triad,
        )

    return F


# ================================================================
# Full symbolic snapshot generators
#
# Each G_m contains:
#
#   4 active pairwise edges
#   +
#   2 persistent native triads
# ================================================================

snapshot_fields_symbolic = []


for snapshot in snapshots:

    pairwise_part = sum(
        (
            edge_fields_symbolic[
                edge
            ]

            for edge
            in snapshot
        ),
        sp.zeros(
            N,
            1,
        ),
    )

    full_field = (
        pairwise_part
        +
        native_total_symbolic
    )

    snapshot_fields_symbolic.append(
        sp.Matrix(
            [
                sp.expand(
                    component
                )

                for component
                in full_field
            ]
        )
    )


# ================================================================
# Numerical snapshot generator
# ================================================================

def snapshot_field_numeric(
    X,
    snapshot_index,
):
    """
    Evaluate instantaneous generator G_m.

    Parameters
    ----------
    X
        State or batch with shape (..., N).

    snapshot_index
        Python index 0,...,5.
    """

    snapshot = snapshots[
        snapshot_index
    ]

    F = np.zeros_like(
        X,
        dtype=float,
    )

    # Temporally active pairwise sector
    for edge in snapshot:

        F += edge_field_numeric(
            X,
            edge,
        )

    # Persistent native HOI sector
    F += native_total_numeric(
        X
    )

    return F


# ================================================================
# Physical consistency audit
#
# Use a separate RNG so this diagnostic does NOT consume the
# trajectory RNG reserved for Cell 2.
# ================================================================

audit_rng = np.random.default_rng(
    SEED + 303
)

X_check = audit_rng.uniform(
    -0.5,
    0.5,
    size=(
        32,
        N,
    ),
)


# Every microscopic symbolic field conserves total state.
for edge, field in (
    edge_fields_symbolic.items()
):

    assert (
        sp.simplify(
            sum(field)
        )
        ==
        0
    )


for triad, field in (
    native_triad_fields_symbolic.items()
):

    assert (
        sp.simplify(
            sum(field)
        )
        ==
        0
    )


# Every full snapshot generator also conserves total state.
max_conservation_error = 0.0


for snapshot_index in range(
    len(snapshots)
):

    F_check = (
        snapshot_field_numeric(
            X_check,
            snapshot_index,
        )
    )

    error = float(
        np.max(
            np.abs(
                F_check.sum(
                    axis=1
                )
            )
        )
    )

    max_conservation_error = max(
        max_conservation_error,
        error,
    )


assert (
    max_conservation_error
    <
    1e-13
)


# ================================================================
# Exact instantaneous structural-support oracle
#
# Used ONLY later for synthetic scoring.
#
# For a monomial appearing in output i:
#
#       support =
#           {output i}
#           union
#           {variables in monomial}
#
# This is exactly the TSC structural-support convention.
# ================================================================

def symbolic_structural_supports(
    field,
):

    supports = set()

    for output_index, expr in enumerate(
        field
    ):

        poly = sp.Poly(
            sp.expand(
                expr
            ),
            *x_symbols,
        )

        for powers, coefficient in (
            poly.terms()
        ):

            if (
                sp.simplify(
                    coefficient
                )
                ==
                0
            ):
                continue

            variables = {
                j + 1

                for j, power
                in enumerate(
                    powers
                )

                if power > 0
            }

            support = tuple(
                sorted(
                    variables
                    |
                    {
                        output_index
                        + 1
                    }
                )
            )

            supports.add(
                support
            )

    return tuple(
        sorted(
            supports,
            key=lambda s: (
                len(s),
                s,
            ),
        )
    )


SNAPSHOT_ORACLE_SUPPORTS = {
    snapshot_index:
        symbolic_structural_supports(
            snapshot_fields_symbolic[
                snapshot_index
            ]
        )

    for snapshot_index
    in range(
        len(snapshots)
    )
}


SNAPSHOT_ORACLE_COUNTS = {
    snapshot_index:
        Counter(
            len(support)

            for support
            in supports
        )

    for snapshot_index, supports
    in SNAPSHOT_ORACLE_SUPPORTS.items()
}


# ================================================================
# Report
# ================================================================

print("=" * 70)
print("FROZEN STAGE-4 PHYSICAL SYSTEM")
print("=" * 70)

print(
    f"Nodes                       : "
    f"{N}"
)

print(
    f"Microscopic pair edges      : "
    f"{len(microscopic_edges)}"
)

print(
    f"Temporal snapshots          : "
    f"{len(snapshots)}"
)

print(
    f"Active pair edges / snapshot: "
    f"{len(snapshots[0])}"
)

print(
    f"Persistent native triads    : "
    f"{NATIVE_TRIADS}"
)

print(
    f"Native triad strength       : "
    f"{NATIVE_G}"
)

print(
    f"Max conservation error      : "
    f"{max_conservation_error:.3e}"
)


print()
print("Frozen edge weights")
print("-" * 70)

for edge in microscopic_edges:

    print(
        f"{edge}: "
        f"{edge_weight[edge]:.3f}"
    )


print()
print("Temporal protocol")
print("-" * 70)

for m, snapshot in enumerate(
    snapshots,
    start=1,
):

    print(
        f"G{m}: "
        f"{snapshot}"
    )


print()
print("Instantaneous support oracle")
print("-" * 70)

for snapshot_index in range(
    len(snapshots)
):

    supports = (
        SNAPSHOT_ORACLE_SUPPORTS[
            snapshot_index
        ]
    )

    counts = (
        SNAPSHOT_ORACLE_COUNTS[
            snapshot_index
        ]
    )

    print(
        f"G{snapshot_index + 1}: "
        f"{len(supports):2d} supports "
        f"| by size = "
        f"{dict(sorted(counts.items()))}"
    )


# Frozen checks
EXPECTED_SNAPSHOT_COUNTS = (
    {1: 8, 2: 9, 3: 2},
    {1: 8, 2: 9, 3: 2},
    {1: 6, 2: 9, 3: 2},
    {1: 7, 2: 9, 3: 2},
    {1: 7, 2: 9, 3: 2},
    {1: 6, 2: 9, 3: 2},
)


for snapshot_index, expected in enumerate(
    EXPECTED_SNAPSHOT_COUNTS
):

    assert (
        dict(
            SNAPSHOT_ORACLE_COUNTS[
                snapshot_index
            ]
        )
        ==
        expected
    )


print()
print("=" * 70)
print("Cell 1 PASSED.")
print("=" * 70)

FROZEN STAGE-4 PHYSICAL SYSTEM
Nodes                       : 8
Microscopic pair edges      : 12
Temporal snapshots          : 6
Active pair edges / snapshot: 4
Persistent native triads    : ((1, 2, 3), (2, 5, 8))
Native triad strength       : 0.02
Max conservation error      : 2.776e-16

Frozen edge weights
----------------------------------------------------------------------
(1, 2): 1.006
(2, 3): 0.911
(3, 4): 1.105
(4, 5): 0.917
(5, 6): 1.177
(6, 7): 1.119
(7, 8): 0.917
(8, 1): 0.944
(1, 3): 1.026
(2, 4): 1.002
(3, 5): 1.067
(5, 7): 1.031

Temporal protocol
----------------------------------------------------------------------
G1: ((1, 2), (3, 4), (5, 6), (7, 8))
G2: ((2, 3), (4, 5), (6, 7), (8, 1))
G3: ((1, 3), (2, 4), (3, 5), (5, 7))
G4: ((1, 2), (4, 5), (7, 8), (3, 5))
G5: ((2, 3), (5, 6), (8, 1), (5, 7))
G6: ((3, 4), (6, 7), (1, 3), (2, 4))

Instantaneous support oracle
----------------------------------------------------------------------
G1: 19 supports | by size = {1: 8, 2: 9

In [3]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# Cell 2: generate ONE continuous repeated-snapshot trajectory
#
# Physical protocol:
#
#   G1 -> G2 -> G3 -> G4 -> G5 -> G6
#      -> G1 -> G2 -> ...  (6 cycles total)
#
# IMPORTANT
# ---------------------------------------------------------------
# There is exactly ONE initial condition.
#
# After this cell, every endpoint pair used for inference must be
# READ from this stored trajectory.
#
# We will NOT restart the simulator from individual slice starts.
# ================================================================

import time


# ================================================================
# Reinitialize trajectory RNG explicitly
#
# Same Stage-4 data seed:
#
#       SEED + 202
#
# This guarantees that previous diagnostic random draws cannot
# affect the formal trajectory initial condition.
# ================================================================

TRAJECTORY_SEED = SEED + 202

trajectory_rng = np.random.default_rng(
    TRAJECTORY_SEED
)


# ================================================================
# Single initial condition
#
# Same state range as the original Stage-4 benchmark.
# ================================================================

X_TRAJ_INITIAL = trajectory_rng.uniform(
    -0.5,
    0.5,
    size=N,
)


# ================================================================
# One fixed RK4 step
#
# dt = DT_OBS = 5e-4
#
# This is finer than the original Stage-4 maximum RK4 step
# of 1e-3.
# ================================================================

def rk4_step_snapshot(
    x,
    snapshot_index,
    dt=DT_OBS,
):
    x = np.asarray(
        x,
        dtype=float,
    )

    k1 = snapshot_field_numeric(
        x,
        snapshot_index,
    )

    k2 = snapshot_field_numeric(
        x + 0.5 * dt * k1,
        snapshot_index,
    )

    k3 = snapshot_field_numeric(
        x + 0.5 * dt * k2,
        snapshot_index,
    )

    k4 = snapshot_field_numeric(
        x + dt * k3,
        snapshot_index,
    )

    return (
        x
        +
        (dt / 6.0)
        * (
            k1
            + 2.0 * k2
            + 2.0 * k3
            + k4
        )
    )


# ================================================================
# Allocate the ONE observed trajectory
#
# TRAJECTORY_STATES[k] = X(t_k)
#
# with
#
#       t_k = k * DT_OBS
#
# Shape:
#
#       (TOTAL_STEPS + 1, N)
# ================================================================

TRAJECTORY_TIMES = (
    np.arange(
        TOTAL_STEPS + 1,
        dtype=float,
    )
    *
    DT_OBS
)

TRAJECTORY_STATES = np.empty(
    (
        TOTAL_STEPS + 1,
        N,
    ),
    dtype=float,
)

TRAJECTORY_STATES[0] = (
    X_TRAJ_INITIAL
)


# ================================================================
# Metadata for every integration interval
#
# STEP_SNAPSHOT[k] tells us which generator acts on:
#
#       [t_k, t_{k+1})
#
# STEP_CYCLE[k] tells us which repeated protocol cycle it belongs to.
# ================================================================

STEP_SNAPSHOT = np.empty(
    TOTAL_STEPS,
    dtype=int,
)

STEP_CYCLE = np.empty(
    TOTAL_STEPS,
    dtype=int,
)


# ================================================================
# Generate the continuous trajectory
# ================================================================

tic = time.perf_counter()

for step in range(
    TOTAL_STEPS
):

    cycle_index = (
        step
        //
        CYCLE_STEPS
    )

    step_inside_cycle = (
        step
        %
        CYCLE_STEPS
    )

    snapshot_index = (
        step_inside_cycle
        //
        SNAPSHOT_STEPS
    )

    assert (
        0
        <= snapshot_index
        < 6
    )

    STEP_CYCLE[
        step
    ] = cycle_index

    STEP_SNAPSHOT[
        step
    ] = snapshot_index

    TRAJECTORY_STATES[
        step + 1
    ] = rk4_step_snapshot(
        TRAJECTORY_STATES[
            step
        ],
        snapshot_index,
        DT_OBS,
    )


TRAJECTORY_RUNTIME = (
    time.perf_counter()
    -
    tic
)


# ================================================================
# Fundamental integrity checks
# ================================================================

assert TRAJECTORY_TIMES.shape == (
    TOTAL_STEPS + 1,
)

assert TRAJECTORY_STATES.shape == (
    TOTAL_STEPS + 1,
    N,
)

assert STEP_SNAPSHOT.shape == (
    TOTAL_STEPS,
)

assert STEP_CYCLE.shape == (
    TOTAL_STEPS,
)

assert np.all(
    np.isfinite(
        TRAJECTORY_STATES
    )
)

assert np.isclose(
    TRAJECTORY_TIMES[-1],
    TOTAL_TIME,
)


# ================================================================
# Exact protocol-count audit
#
# Every cycle must contain exactly:
#
#       120 steps of G1
#       ...
#       120 steps of G6
# ================================================================

for cycle_index in range(
    N_CYCLES
):

    cycle_mask = (
        STEP_CYCLE
        ==
        cycle_index
    )

    counts = Counter(
        STEP_SNAPSHOT[
            cycle_mask
        ]
    )

    assert counts == {
        snapshot_index:
            SNAPSHOT_STEPS

        for snapshot_index
        in range(6)
    }


# ================================================================
# Conservation audit
#
# Every microscopic field conserves sum_i x_i, so the entire
# switched trajectory must conserve the same total mass.
# ================================================================

INITIAL_MASS = float(
    X_TRAJ_INITIAL.sum()
)

TRAJECTORY_MASS = (
    TRAJECTORY_STATES.sum(
        axis=1
    )
)

MAX_TRAJECTORY_MASS_ERROR = float(
    np.max(
        np.abs(
            TRAJECTORY_MASS
            -
            INITIAL_MASS
        )
    )
)

assert (
    MAX_TRAJECTORY_MASS_ERROR
    <
    1e-11
)


# ================================================================
# Dynamical-excitation audit
#
# This matters specifically for one-time-series inference.
#
# Because the network is conservative but mixing/dissipative,
# the trajectory may progressively approach consensus.
#
# Define:
#
#   node_spread(t)
#       = RMS deviation of node states around their instantaneous
#         node mean.
#
# Since mass is conserved, the node mean is constant.
# ================================================================

TRAJECTORY_NODE_MEAN = (
    TRAJECTORY_STATES.mean(
        axis=1
    )
)

TRAJECTORY_NODE_SPREAD = np.sqrt(
    np.mean(
        (
            TRAJECTORY_STATES
            -
            TRAJECTORY_NODE_MEAN[
                :, None
            ]
        )
        ** 2,
        axis=1,
    )
)


# ================================================================
# Per-cycle state-space diagnostics
# ================================================================

cycle_rows = []

for cycle_index in range(
    N_CYCLES
):

    start_step = (
        cycle_index
        *
        CYCLE_STEPS
    )

    end_step = (
        (cycle_index + 1)
        *
        CYCLE_STEPS
    )

    X_cycle = (
        TRAJECTORY_STATES[
            start_step:
            end_step + 1
        ]
    )

    spread_cycle = (
        TRAJECTORY_NODE_SPREAD[
            start_step:
            end_step + 1
        ]
    )

    state_excursion = (
        np.sqrt(
            np.mean(
                (
                    X_cycle
                    -
                    X_cycle[0]
                )
                ** 2
            )
        )
    )

    cycle_rows.append({
        "cycle":
            cycle_index,

        "t_start":
            TRAJECTORY_TIMES[
                start_step
            ],

        "t_end":
            TRAJECTORY_TIMES[
                end_step
            ],

        "spread_start":
            spread_cycle[0],

        "spread_end":
            spread_cycle[-1],

        "spread_mean":
            spread_cycle.mean(),

        "state_excursion_rms":
            state_excursion,

        "state_min":
            X_cycle.min(),

        "state_max":
            X_cycle.max(),
    })


TRAJECTORY_CYCLE_AUDIT = pd.DataFrame(
    cycle_rows
)


# ================================================================
# Snapshot-boundary state table
#
# Useful later when we diagnose inference near switching points.
# ================================================================

boundary_rows = []

for cycle_index in range(
    N_CYCLES
):

    for snapshot_index in range(
        6
    ):

        step = (
            cycle_index
            *
            CYCLE_STEPS

            +
            snapshot_index
            *
            SNAPSHOT_STEPS
        )

        boundary_rows.append({
            "cycle":
                cycle_index,

            "snapshot":
                snapshot_index + 1,

            "step":
                step,

            "time":
                TRAJECTORY_TIMES[
                    step
                ],

            "node_spread":
                TRAJECTORY_NODE_SPREAD[
                    step
                ],
        })


SNAPSHOT_BOUNDARY_TABLE = pd.DataFrame(
    boundary_rows
)


# ================================================================
# Report
# ================================================================

print("=" * 70)
print("ONE CONTINUOUS N=8 TRAJECTORY")
print("=" * 70)

print(
    f"trajectory seed             : "
    f"{TRAJECTORY_SEED}"
)

print(
    f"initial state               : "
    f"{np.array2string(
        X_TRAJ_INITIAL,
        precision=6
    )}"
)

print(
    f"initial mass                : "
    f"{INITIAL_MASS:.12f}"
)

print(
    f"time interval               : "
    f"[0, {TOTAL_TIME}]"
)

print(
    f"observation interval        : "
    f"{DT_OBS}"
)

print(
    f"observed states             : "
    f"{len(TRAJECTORY_STATES)}"
)

print(
    f"integration intervals       : "
    f"{TOTAL_STEPS}"
)

print(
    f"runtime                     : "
    f"{TRAJECTORY_RUNTIME:.3f} s"
)

print(
    f"max conservation error      : "
    f"{MAX_TRAJECTORY_MASS_ERROR:.3e}"
)

print(
    f"global state range          : "
    f"[{TRAJECTORY_STATES.min():.6f}, "
    f"{TRAJECTORY_STATES.max():.6f}]"
)

print(
    f"node spread: start -> end   : "
    f"{TRAJECTORY_NODE_SPREAD[0]:.6e} "
    f"-> "
    f"{TRAJECTORY_NODE_SPREAD[-1]:.6e}"
)

print(
    f"spread retention            : "
    f"{TRAJECTORY_NODE_SPREAD[-1] /
       TRAJECTORY_NODE_SPREAD[0]:.6f}"
)


print()
print("Per-cycle excitation audit")
print("-" * 70)

display(
    TRAJECTORY_CYCLE_AUDIT.style.format({
        "t_start":
            "{:.3f}",

        "t_end":
            "{:.3f}",

        "spread_start":
            "{:.6e}",

        "spread_end":
            "{:.6e}",

        "spread_mean":
            "{:.6e}",

        "state_excursion_rms":
            "{:.6e}",

        "state_min":
            "{:.6f}",

        "state_max":
            "{:.6f}",
    })
)


# ================================================================
# Save the ONE observed time series
#
# Ground-truth snapshot labels are saved separately as synthetic
# oracle metadata. They will not be supplied to the learner in
# the later unlabelled experiment.
# ================================================================

TRAJECTORY_FILE = (
    OUTPUT_DIR
    /
    "n8_single_continuous_trajectory.npz"
)

np.savez_compressed(
    TRAJECTORY_FILE,

    times=
        TRAJECTORY_TIMES,

    states=
        TRAJECTORY_STATES,

    dt_obs=
        np.array(
            DT_OBS
        ),

    initial_state=
        X_TRAJ_INITIAL,

    trajectory_seed=
        np.array(
            TRAJECTORY_SEED
        ),

    # Synthetic oracle metadata
    step_snapshot=
        STEP_SNAPSHOT,

    step_cycle=
        STEP_CYCLE,
)


TRAJECTORY_CYCLE_AUDIT.to_csv(
    OUTPUT_DIR
    /
    "n8_single_trajectory_cycle_audit.csv",
    index=False,
)


print()
print(
    f"Saved trajectory            : "
    f"{TRAJECTORY_FILE}"
)

print()
print("=" * 70)
print("Cell 2 PASSED.")
print("=" * 70)

ONE CONTINUOUS N=8 TRAJECTORY
trajectory seed             : 20261013
initial state               : [-0.140814 -0.264819  0.181982 -0.390419  0.272227 -0.116231 -0.228057
 -0.418635]
initial mass                : -1.104766581081
time interval               : [0, 2.16]
observation interval        : 0.0005
observed states             : 4321
integration intervals       : 4320
runtime                     : 4.475 s
max conservation error      : 1.110e-15
global state range          : [-0.418635, 0.272227]
node spread: start -> end   : 2.339393e-01 -> 5.001122e-02
spread retention            : 0.213779

Per-cycle excitation audit
----------------------------------------------------------------------


,cycle,t_start,t_end,spread_start,spread_end,spread_mean,state_excursion_rms,state_min,state_max
0,0,0.000,0.360,2.339393e-01,1.585989e-01,1.923930e-01,5.179997e-02,-0.418635,0.272227
1,1,0.360,0.720,1.585989e-01,1.152496e-01,1.347842e-01,3.084650e-02,-0.384521,0.120355
2,2,0.720,1.080,1.152496e-01,8.881113e-02,1.007675e-01,1.937449e-02,-0.351540,0.033930
3,3,1.080,1.440,8.881113e-02,7.149441e-02,7.934365e-02,1.283483e-02,-0.321530,-0.017926
4,4,1.440,1.800,7.149441e-02,5.925699e-02,6.480398e-02,8.993320e-03,-0.295087,-0.050432
5,5,1.800,2.160,5.925699e-02,5.001122e-02,5.419280e-02,6.659965e-03,-0.272202,-0.071664



Saved trajectory            : stage6_n8_timeseries_slicing\n8_single_continuous_trajectory.npz

Cell 2 PASSED.


In [11]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# NEW Cell 3:
# topology-agnostic interaction library
# + clean Case-2 observable dataset
#
# RESEARCH RESET
# ---------------------------------------------------------------
# From this cell onward the inverse problem is:
#
#     single observed trajectory
#           ->
#     finite-window transitions
#           ->
#     effective temporal network
#
# The learner is allowed to know:
#
#   - N
#   - DeltaT
#   - microscopic interaction TYPES / functional forms
#   - all possible pair / triad supports
#
# The learner is NOT allowed to know:
#
#   - which microscopic edges are active
#   - which native triads are active
#   - G1 or G2
#   - microscopic coefficients
#   - the BCH oracle
#
# ================================================================

import numpy as np
import pandas as pd
from itertools import combinations


# ================================================================
# 0. Admissible microscopic interaction class
#
# All possible pair supports:
#
#       C(8,2) = 28
#
# All possible triad supports:
#
#       C(8,3) = 56
#
# Total:
#
#       Q = 84
#
# This is topology agnostic.
# ================================================================

RS_PAIR_SUPPORTS = tuple(
    combinations(
        range(1, N + 1),
        2,
    )
)

RS_TRIAD_SUPPORTS = tuple(
    combinations(
        range(1, N + 1),
        3,
    )
)

RS_MICRO_SUPPORTS = (
    RS_PAIR_SUPPORTS
    +
    RS_TRIAD_SUPPORTS
)

RS_Q_PAIR = len(
    RS_PAIR_SUPPORTS
)

RS_Q_TRIAD = len(
    RS_TRIAD_SUPPORTS
)

RS_Q = len(
    RS_MICRO_SUPPORTS
)


assert RS_Q_PAIR == 28
assert RS_Q_TRIAD == 56
assert RS_Q == 84


RS_MICRO_META = pd.DataFrame({

    "q":
        np.arange(
            RS_Q,
            dtype=int,
        ),

    "kind":
        (
            ["pair"] * RS_Q_PAIR
            +
            ["triad"] * RS_Q_TRIAD
        ),

    "support":
        RS_MICRO_SUPPORTS,

    "support_order":
        [
            len(s)
            for s in RS_MICRO_SUPPORTS
        ],
})


# ================================================================
# 1. Zero-based support arrays for fast numerical evaluation
# ================================================================

RS_PAIR_INDEX = np.asarray(
    [
        (
            i - 1,
            j - 1,
        )
        for i, j
        in RS_PAIR_SUPPORTS
    ],
    dtype=int,
)

RS_TRIAD_INDEX = np.asarray(
    [
        (
            i - 1,
            j - 1,
            k - 1,
        )
        for i, j, k
        in RS_TRIAD_SUPPORTS
    ],
    dtype=int,
)


RS_PAIR_I = RS_PAIR_INDEX[:, 0]
RS_PAIR_J = RS_PAIR_INDEX[:, 1]

RS_TRIAD_I = RS_TRIAD_INDEX[:, 0]
RS_TRIAD_J = RS_TRIAD_INDEX[:, 1]
RS_TRIAD_K = RS_TRIAD_INDEX[:, 2]


RS_PAIR_Q = np.arange(
    RS_Q_PAIR,
    dtype=int,
)

RS_TRIAD_Q = (
    RS_Q_PAIR
    +
    np.arange(
        RS_Q_TRIAD,
        dtype=int,
    )
)


# ================================================================
# 2. Unit microscopic vector-field basis
#
# Pair candidate Psi_ij:
#
#   Psi_i = phi(x_j) - phi(x_i)
#   Psi_j = -Psi_i
#
# Its unknown coefficient therefore carries the edge strength.
#
#
# Triad candidate Psi_ijk:
#
# same native triad interaction law as Cell 1,
# but UNIT strength g = 1.
#
# Its unknown coefficient carries the triad strength.
#
#
# Output:
#
#       B(x) : shape (N,84)
#
#       F_beta(x) = B(x) @ beta
# ================================================================

def rs_basis_matrix(
    x,
):

    x = np.asarray(
        x,
        dtype=float,
    )

    assert x.shape == (N,)


    B = np.zeros(
        (
            N,
            RS_Q,
        ),
        dtype=float,
    )


    # ------------------------------------------------------------
    # Pair sector
    # ------------------------------------------------------------

    xi = x[
        RS_PAIR_I
    ]

    xj = x[
        RS_PAIR_J
    ]

    pair_value = (
        phi_numeric(
            xj
        )
        -
        phi_numeric(
            xi
        )
    )


    B[
        RS_PAIR_I,
        RS_PAIR_Q
    ] = pair_value

    B[
        RS_PAIR_J,
        RS_PAIR_Q
    ] = -pair_value


    # ------------------------------------------------------------
    # Triad sector
    # ------------------------------------------------------------

    xi = x[
        RS_TRIAD_I
    ]

    xj = x[
        RS_TRIAD_J
    ]

    xk = x[
        RS_TRIAD_K
    ]


    Fi = (
        xj * xk
        -
        0.5 * xi * xj
        -
        0.5 * xi * xk
    )

    Fj = (
        xi * xk
        -
        0.5 * xj * xi
        -
        0.5 * xj * xk
    )

    Fk = (
        xi * xj
        -
        0.5 * xk * xi
        -
        0.5 * xk * xj
    )


    B[
        RS_TRIAD_I,
        RS_TRIAD_Q
    ] = Fi

    B[
        RS_TRIAD_J,
        RS_TRIAD_Q
    ] = Fj

    B[
        RS_TRIAD_K,
        RS_TRIAD_Q
    ] = Fk


    return B


# ================================================================
# 3. Analytic Jacobians of all 84 basis fields
#
# Output:
#
#       J(x) : shape (N,N,84)
#
#       J[:,:,q] = D Psi_q(x)
# ================================================================

def rs_basis_jacobians(
    x,
):

    x = np.asarray(
        x,
        dtype=float,
    )

    assert x.shape == (N,)


    J = np.zeros(
        (
            N,
            N,
            RS_Q,
        ),
        dtype=float,
    )


    # ------------------------------------------------------------
    # Pair sector
    #
    # phi'(x) = 1 + 2 lambda x
    # ------------------------------------------------------------

    xi = x[
        RS_PAIR_I
    ]

    xj = x[
        RS_PAIR_J
    ]


    dpi = (
        1.0
        +
        2.0 * LAMBDA * xi
    )

    dpj = (
        1.0
        +
        2.0 * LAMBDA * xj
    )


    J[
        RS_PAIR_I,
        RS_PAIR_I,
        RS_PAIR_Q
    ] = -dpi

    J[
        RS_PAIR_I,
        RS_PAIR_J,
        RS_PAIR_Q
    ] = dpj

    J[
        RS_PAIR_J,
        RS_PAIR_I,
        RS_PAIR_Q
    ] = dpi

    J[
        RS_PAIR_J,
        RS_PAIR_J,
        RS_PAIR_Q
    ] = -dpj


    # ------------------------------------------------------------
    # Triad sector
    # ------------------------------------------------------------

    i = RS_TRIAD_I
    j = RS_TRIAD_J
    k = RS_TRIAD_K
    q = RS_TRIAD_Q


    xi = x[i]
    xj = x[j]
    xk = x[k]


    # F_i derivatives
    J[
        i,
        i,
        q
    ] = -0.5 * (
        xj + xk
    )

    J[
        i,
        j,
        q
    ] = (
        xk
        -
        0.5 * xi
    )

    J[
        i,
        k,
        q
    ] = (
        xj
        -
        0.5 * xi
    )


    # F_j derivatives
    J[
        j,
        i,
        q
    ] = (
        xk
        -
        0.5 * xj
    )

    J[
        j,
        j,
        q
    ] = -0.5 * (
        xi + xk
    )

    J[
        j,
        k,
        q
    ] = (
        xi
        -
        0.5 * xj
    )


    # F_k derivatives
    J[
        k,
        i,
        q
    ] = (
        xj
        -
        0.5 * xk
    )

    J[
        k,
        j,
        q
    ] = (
        xi
        -
        0.5 * xk
    )

    J[
        k,
        k,
        q
    ] = -0.5 * (
        xi + xj
    )


    return J


# ================================================================
# 4. Aggregate microscopic-class field
# ================================================================

def rs_field_from_beta(
    x,
    beta,
):

    beta = np.asarray(
        beta,
        dtype=float,
    )

    assert beta.shape == (
        RS_Q,
    )

    return (
        rs_basis_matrix(
            x
        )
        @
        beta
    )


def rs_jacobian_from_beta(
    x,
    beta,
):

    beta = np.asarray(
        beta,
        dtype=float,
    )

    assert beta.shape == (
        RS_Q,
    )

    return np.einsum(
        "ijq,q->ij",
        rs_basis_jacobians(
            x
        ),
        beta,
        optimize=True,
    )


# ================================================================
# 5. Library sanity audit
#
# This checks only the admissible interaction class.
# It uses no true topology.
# ================================================================

RS_LIBRARY_RNG = np.random.default_rng(
    SEED + 404
)

RS_LIBRARY_PROBES = (
    RS_LIBRARY_RNG.uniform(
        -0.6,
        0.6,
        size=(
            64,
            N,
        ),
    )
)


RS_LIBRARY_DESIGN = np.vstack(
    [
        rs_basis_matrix(
            x
        )
        for x in RS_LIBRARY_PROBES
    ]
)


RS_LIBRARY_RANK = (
    np.linalg.matrix_rank(
        RS_LIBRARY_DESIGN,
        tol=1e-11,
    )
)


RS_LIBRARY_CONSERVATION_ERROR = max(

    float(
        np.max(
            np.abs(
                rs_basis_matrix(
                    x
                ).sum(
                    axis=0
                )
            )
        )
    )

    for x in RS_LIBRARY_PROBES
)


assert RS_LIBRARY_RANK == RS_Q

assert (
    RS_LIBRARY_CONSERVATION_ERROR
    <
    1e-12
)


# ================================================================
# 6. Clean Case-2 finite-window experiment
#
# Center window:
#
#       [0.045, 0.075]
#
# relative to the start of every six-snapshot cycle.
#
# It crosses exactly one microscopic switching boundary:
#
#       G1 for 0.015
#       G2 for 0.015
#
# BUT this fact will NOT be supplied to the learner.
#
#
# Local-C ensemble:
#
#       delta = -0.0005, 0, +0.0005
#
# All endpoint states are READ from the continuous trajectory.
# No reintegration.
# ================================================================

RS_TARGET_START = 0.045

RS_TARGET_DT = 0.030

RS_LOCAL_OFFSETS = np.array(
    [
        -DT_OBS,
        0.0,
        +DT_OBS,
    ],
    dtype=float,
)


RS_TARGET_START_STEP = integer_steps(
    RS_TARGET_START
)

RS_TARGET_DT_STEPS = integer_steps(
    RS_TARGET_DT
)

RS_LOCAL_OFFSET_STEPS = np.array(
    [
        integer_steps(
            abs(offset)
        )
        *
        (
            -1
            if offset < 0
            else
            1
        )

        for offset in RS_LOCAL_OFFSETS
    ],
    dtype=int,
)


assert np.array_equal(
    RS_LOCAL_OFFSET_STEPS,
    np.array(
        [-1, 0, 1],
        dtype=int,
    ),
)


# ================================================================
# 7. Slice trajectory
# ================================================================

rs_rows = []

rs_X0 = []
rs_XF = []


for cycle in range(
    N_CYCLES
):

    cycle_start_step = (
        cycle
        *
        CYCLE_STEPS
    )


    if cycle in FIT_CYCLES:

        role = "fit"

    elif cycle in VALIDATION_CYCLES:

        role = "validation"

    elif cycle in TEST_CYCLES:

        role = "test"

    else:

        raise RuntimeError(
            "Unexpected cycle assignment."
        )


    for (
        offset,
        offset_step,
    ) in zip(
        RS_LOCAL_OFFSETS,
        RS_LOCAL_OFFSET_STEPS,
    ):

        start_step = (
            cycle_start_step
            +
            RS_TARGET_START_STEP
            +
            offset_step
        )

        end_step = (
            start_step
            +
            RS_TARGET_DT_STEPS
        )


        assert (
            cycle_start_step
            <=
            start_step
        )

        assert (
            end_step
            <=
            cycle_start_step
            +
            CYCLE_STEPS
        )


        X0 = (
            TRAJECTORY_STATES[
                start_step
            ]
            .copy()
        )

        XF = (
            TRAJECTORY_STATES[
                end_step
            ]
            .copy()
        )


        rs_X0.append(
            X0
        )

        rs_XF.append(
            XF
        )


        rs_rows.append({

            "cycle":
                cycle,

            "role":
                role,

            "offset":
                float(
                    offset
                ),

            "start_step":
                int(
                    start_step
                ),

            "end_step":
                int(
                    end_step
                ),

            "start_time":
                float(
                    TRAJECTORY_TIMES[
                        start_step
                    ]
                ),

            "end_time":
                float(
                    TRAJECTORY_TIMES[
                        end_step
                    ]
                ),
        })


RS_CASE2_META = pd.DataFrame(
    rs_rows
)

RS_CASE2_X0 = np.asarray(
    rs_X0,
    dtype=float,
)

RS_CASE2_XF = np.asarray(
    rs_XF,
    dtype=float,
)


assert RS_CASE2_X0.shape == (
    18,
    N,
)

assert RS_CASE2_XF.shape == (
    18,
    N,
)


# ================================================================
# 8. Split masks
# ================================================================

RS_CASE2_FIT_MASK = (
    RS_CASE2_META[
        "role"
    ].to_numpy()
    ==
    "fit"
)

RS_CASE2_VAL_MASK = (
    RS_CASE2_META[
        "role"
    ].to_numpy()
    ==
    "validation"
)

RS_CASE2_TEST_MASK = (
    RS_CASE2_META[
        "role"
    ].to_numpy()
    ==
    "test"
)

RS_CASE2_CENTER_MASK = np.isclose(
    RS_CASE2_META[
        "offset"
    ].to_numpy(),
    0.0,
    atol=1e-15,
)


assert (
    RS_CASE2_FIT_MASK.sum()
    ==
    12
)

assert (
    RS_CASE2_VAL_MASK.sum()
    ==
    3
)

assert (
    RS_CASE2_TEST_MASK.sum()
    ==
    3
)

assert (
    RS_CASE2_CENTER_MASK.sum()
    ==
    6
)


# ================================================================
# 9. Observable-only object supplied to the solver
#
# NOTE:
# No snapshot labels, G1/G2 fractions, true supports, or oracle
# coefficients are present here.
# ================================================================

RS_CASE2_DIRECT_DATA = {

    "DeltaT":
        RS_TARGET_DT,

    "X0_fit":
        RS_CASE2_X0[
            RS_CASE2_FIT_MASK
        ],

    "XF_fit":
        RS_CASE2_XF[
            RS_CASE2_FIT_MASK
        ],

    "X0_validation":
        RS_CASE2_X0[
            RS_CASE2_VAL_MASK
        ],

    "XF_validation":
        RS_CASE2_XF[
            RS_CASE2_VAL_MASK
        ],

    "X0_test":
        RS_CASE2_X0[
            RS_CASE2_TEST_MASK
        ],

    "XF_test":
        RS_CASE2_XF[
            RS_CASE2_TEST_MASK
        ],
}


# ================================================================
# 10. Report
# ================================================================

print("=" * 86)
print("NEW RS-TSC STUDY — CELL 3")
print("ADMISSIBLE INTERACTION CLASS + CLEAN CASE-2 DATA")
print("=" * 86)

print()
print("Interaction class")
print("-" * 86)

print(
    f"possible pair channels       : "
    f"{RS_Q_PAIR}"
)

print(
    f"possible triad channels      : "
    f"{RS_Q_TRIAD}"
)

print(
    f"total microscopic channels   : "
    f"{RS_Q}"
)

print(
    f"generic library rank         : "
    f"{RS_LIBRARY_RANK}"
)

print(
    f"conservation error           : "
    f"{RS_LIBRARY_CONSERVATION_ERROR:.3e}"
)


print()
print("Finite-window dataset")
print("-" * 86)

print(
    f"target relative start        : "
    f"{RS_TARGET_START:.6f}"
)

print(
    f"DeltaT                       : "
    f"{RS_TARGET_DT:.6f}"
)

print(
    f"local offsets                : "
    f"{RS_LOCAL_OFFSETS}"
)

print(
    f"fit windows                  : "
    f"{RS_CASE2_FIT_MASK.sum()}"
)

print(
    f"validation windows           : "
    f"{RS_CASE2_VAL_MASK.sum()}"
)

print(
    f"external test windows        : "
    f"{RS_CASE2_TEST_MASK.sum()}"
)

print(
    f"center windows               : "
    f"{RS_CASE2_CENTER_MASK.sum()}"
)


print()
print("Solver-visible information")
print("-" * 86)

print(
    "X0, XF, DeltaT               : YES"
)

print(
    "interaction functional class : YES"
)

print(
    "possible supports            : YES"
)

print(
    "true topology                : NO"
)

print(
    "G1 / G2 labels               : NO"
)

print(
    "BCH oracle                   : NO"
)


print()
print("Window metadata")
print("-" * 86)

display(
    RS_CASE2_META
)


print()
print("=" * 86)
print("Cell 3 PASSED.")
print("=" * 86)

NEW RS-TSC STUDY — CELL 3
ADMISSIBLE INTERACTION CLASS + CLEAN CASE-2 DATA

Interaction class
--------------------------------------------------------------------------------------
possible pair channels       : 28
possible triad channels      : 56
total microscopic channels   : 84
generic library rank         : 84
conservation error           : 1.110e-16

Finite-window dataset
--------------------------------------------------------------------------------------
target relative start        : 0.045000
DeltaT                       : 0.030000
local offsets                : [-0.0005  0.      0.0005]
fit windows                  : 12
validation windows           : 3
external test windows        : 3
center windows               : 6

Solver-visible information
--------------------------------------------------------------------------------------
X0, XF, DeltaT               : YES
interaction functional class : YES
possible supports            : YES
true topology                : NO
G1 / G2 

,cycle,role,offset,start_step,end_step,start_time,end_time
0,0,fit,-0.0005,89,149,0.0445,0.0745
1,0,fit,0.0000,90,150,0.0450,0.0750
2,0,fit,0.0005,91,151,0.0455,0.0755
3,1,fit,-0.0005,809,869,0.4045,0.4345
4,1,fit,0.0000,810,870,0.4050,0.4350
5,1,fit,0.0005,811,871,0.4055,0.4355
6,2,fit,-0.0005,1529,1589,0.7645,0.7945
7,2,fit,0.0000,1530,1590,0.7650,0.7950
8,2,fit,0.0005,1531,1591,0.7655,0.7955
9,3,fit,-0.0005,2249,2309,1.1245,1.1545



Cell 3 PASSED.


In [12]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# NEW Cell 4:
# TSC-constrained effective model
#
# MODEL
# ---------------------------------------------------------------
#
#   G_mu(x) = sum_q mu_q Psi_q(x)
#
#   G_nu(x) = sum_q nu_q Psi_q(x)
#
#   B_chr(G_mu,G_nu)
#       = D G_nu G_mu - D G_mu G_nu
#
#   F_eff(x;mu,nu)
#       = G_mu(x) + B_chr(G_mu,G_nu)(x)
#
#
# INTERPRETATION
# ---------------------------------------------------------------
#
# mu:
#   native / persistent effective coordinates
#
# nu:
#   temporal contrast coordinates, including the finite-window
#   BCH/Magnus prefactor.
#
#
# The first temporal Lie sector therefore has coefficient matrix
#
#   A_ab = mu_a nu_b - nu_a mu_b
#
# which is:
#
#   - antisymmetric
#   - rank <= 2
#   - not an arbitrary collection of bracket coefficients
#
#
# NO true topology or G1/G2 information is used here.
# ================================================================

import numpy as np
import pandas as pd


# ================================================================
# 0. Aggregate fields
# ================================================================

def rs_G(
    x,
    beta,
):

    beta = np.asarray(
        beta,
        dtype=float,
    )

    assert beta.shape == (
        RS_Q,
    )

    return (
        rs_basis_matrix(
            x
        )
        @
        beta
    )


def rs_DG(
    x,
    beta,
):

    beta = np.asarray(
        beta,
        dtype=float,
    )

    assert beta.shape == (
        RS_Q,
    )

    return np.einsum(
        "ijq,q->ij",
        rs_basis_jacobians(
            x
        ),
        beta,
        optimize=True,
    )


# ================================================================
# 1. Chronological Lie bracket
#
# Convention:
#
#   B_chr(F,G)
#       = D G F - D F G
#
# This is the convention previously verified against the actual
# chronological composition:
#
#       first G1, then G2
#
# giving
#
#       D G2 G1 - D G1 G2.
# ================================================================

def rs_temporal_bracket(
    x,
    mu,
    nu,
):

    G_mu = rs_G(
        x,
        mu,
    )

    G_nu = rs_G(
        x,
        nu,
    )

    J_mu = rs_DG(
        x,
        mu,
    )

    J_nu = rs_DG(
        x,
        nu,
    )

    return (
        J_nu @ G_mu
        -
        J_mu @ G_nu
    )


# ================================================================
# 2. TSC-constrained first-order effective generator
# ================================================================

def rs_Feff_rank2(
    x,
    mu,
    nu,
):

    return (
        rs_G(
            x,
            mu,
        )
        +
        rs_temporal_bracket(
            x,
            mu,
            nu,
        )
    )


# ================================================================
# 3. Temporal forcing matrix
#
# Because the Lie bracket is linear in nu when mu is fixed,
#
#     B_chr(G_mu,G_nu)
#         = H_mu(x) @ nu
#
# where column q is
#
#     H_q
#       = D Psi_q G_mu
#         - D G_mu Psi_q.
#
# This object will be central to the solver:
#
# Stage M1 can infer nu from finite-flow endpoint sensitivities
# without treating all pairwise brackets as independent unknowns.
# ================================================================

def rs_temporal_forcing_matrix(
    x,
    mu,
):

    mu = np.asarray(
        mu,
        dtype=float,
    )

    assert mu.shape == (
        RS_Q,
    )


    Psi = rs_basis_matrix(
        x
    )

    J_basis = rs_basis_jacobians(
        x
    )


    G_mu = (
        Psi
        @
        mu
    )

    J_mu = np.einsum(
        "ijq,q->ij",
        J_basis,
        mu,
        optimize=True,
    )


    # D Psi_q @ G_mu
    first = np.einsum(
        "ijq,j->iq",
        J_basis,
        G_mu,
        optimize=True,
    )


    # D G_mu @ Psi_q
    second = (
        J_mu
        @
        Psi
    )


    return (
        first
        -
        second
    )


# ================================================================
# 4. Lie coefficient matrix
#
# Expanding the temporal sector:
#
#   B_chr(G_mu,G_nu)
#
# produces coefficients
#
#   A_ab = mu_a nu_b - nu_a mu_b.
#
# A is a decomposable antisymmetric bivector:
#
#   A = mu ^ nu
#
# and therefore rank(A) <= 2.
# ================================================================

def rs_lie_coefficient_matrix(
    mu,
    nu,
):

    mu = np.asarray(
        mu,
        dtype=float,
    )

    nu = np.asarray(
        nu,
        dtype=float,
    )

    return (
        np.outer(
            mu,
            nu,
        )
        -
        np.outer(
            nu,
            mu,
        )
    )


# ================================================================
# 5. Gauge structure
#
# The transformation
#
#       nu -> nu + c mu
#
# leaves the Lie sector unchanged:
#
#       mu ^ (nu + c mu) = mu ^ nu.
#
# We do NOT impose a gauge here.
#
# Later sparse inference should be free to choose the sparsest
# representative instead of being forced into an orthogonal gauge.
# ================================================================

def rs_orthogonal_temporal_gauge(
    mu,
    nu,
):

    """
    Optional post-hoc representation only.

    Not intended for use during sparse model selection.
    """

    mu = np.asarray(
        mu,
        dtype=float,
    )

    nu = np.asarray(
        nu,
        dtype=float,
    )

    denom = float(
        mu @ mu
    )

    if denom <= np.finfo(float).tiny:

        return nu.copy()


    return (
        nu
        -
        mu
        *
        float(
            mu @ nu
        )
        /
        denom
    )


# ================================================================
# 6. First-Lie-depth structural reachability
#
# We now enumerate which NODE SUPPORTS are algebraically reachable
# at first Lie depth.
#
# Important:
#
# This is NOT topology inference.
#
# It depends only on the admissible interaction class:
#
#       all 28 pair candidates
#       all 56 triad candidates.
#
#
# A bracket can be nonzero only if the parent supports overlap.
#
# Its support is contained in their union.
# ================================================================

rs_lie_rows = []


for a in range(
    RS_Q
):

    Sa = set(
        RS_MICRO_SUPPORTS[a]
    )

    kind_a = (
        RS_MICRO_META.loc[
            a,
            "kind"
        ]
    )


    for b in range(
        a + 1,
        RS_Q,
    ):

        Sb = set(
            RS_MICRO_SUPPORTS[b]
        )

        kind_b = (
            RS_MICRO_META.loc[
                b,
                "kind"
            ]
        )


        # Disjoint vector fields commute structurally.
        if Sa.isdisjoint(
            Sb
        ):

            continue


        support_union = tuple(
            sorted(
                Sa | Sb
            )
        )


        if (
            kind_a == "pair"
            and
            kind_b == "pair"
        ):

            parent_type = (
                "pair-pair"
            )

        elif (
            kind_a == "triad"
            and
            kind_b == "triad"
        ):

            parent_type = (
                "triad-triad"
            )

        else:

            parent_type = (
                "pair-triad"
            )


        rs_lie_rows.append({

            "a":
                a,

            "b":
                b,

            "parent_type":
                parent_type,

            "support":
                support_union,

            "support_order":
                len(
                    support_union
                ),
        })


RS_FIRST_LIE_META = pd.DataFrame(
    rs_lie_rows
)


# ================================================================
# 7. Reachability summary
# ================================================================

RS_FIRST_LIE_SUMMARY = (
    RS_FIRST_LIE_META
    .groupby(
        [
            "parent_type",
            "support_order",
        ]
    )
    .size()
    .rename(
        "n_possible_channels"
    )
    .reset_index()
    .sort_values(
        [
            "support_order",
            "parent_type",
        ]
    )
)


# ================================================================
# 8. Pairwise-only support-depth theorem check
#
# If the admissible microscopic class were pairwise only:
#
#       [2,2] -> at most 3 nodes
#
# therefore first-depth four-body interactions are forbidden.
# ================================================================

RS_PAIRPAIR_ORDER4PLUS = int(
    np.sum(
        (
            RS_FIRST_LIE_META[
                "parent_type"
            ]
            ==
            "pair-pair"
        )
        &
        (
            RS_FIRST_LIE_META[
                "support_order"
            ]
            >=
            4
        )
    )
)


assert (
    RS_PAIRPAIR_ORDER4PLUS
    ==
    0
)


# ================================================================
# 9. Pure model-level numerical sanity checks
#
# Random coefficients are used deliberately.
#
# These checks contain no synthetic-system oracle information.
# ================================================================

RS_MODEL_RNG = np.random.default_rng(
    SEED + 505
)


RS_MODEL_X = (
    RS_MODEL_RNG.uniform(
        -0.5,
        0.7,
        size=N,
    )
)


RS_MODEL_MU = (
    RS_MODEL_RNG.normal(
        0.0,
        0.3,
        size=RS_Q,
    )
)


RS_MODEL_NU = (
    RS_MODEL_RNG.normal(
        0.0,
        0.05,
        size=RS_Q,
    )
)


# ---------------------------------------------------------------
# 9a. Forcing-matrix equivalence
#
#     bracket(mu,nu)
#       == H_mu @ nu
# ---------------------------------------------------------------

RS_BRACKET_DIRECT = (
    rs_temporal_bracket(
        RS_MODEL_X,
        RS_MODEL_MU,
        RS_MODEL_NU,
    )
)


RS_BRACKET_FORCING = (
    rs_temporal_forcing_matrix(
        RS_MODEL_X,
        RS_MODEL_MU,
    )
    @
    RS_MODEL_NU
)


RS_FORCING_ERROR = (
    np.linalg.norm(
        RS_BRACKET_DIRECT
        -
        RS_BRACKET_FORCING
    )
    /
    max(
        np.linalg.norm(
            RS_BRACKET_DIRECT
        ),
        np.finfo(float).tiny,
    )
)


# ---------------------------------------------------------------
# 9b. Gauge invariance
# ---------------------------------------------------------------

RS_GAUGE_C = 1.731


RS_F_GAUGE_0 = (
    rs_Feff_rank2(
        RS_MODEL_X,
        RS_MODEL_MU,
        RS_MODEL_NU,
    )
)


RS_F_GAUGE_1 = (
    rs_Feff_rank2(
        RS_MODEL_X,
        RS_MODEL_MU,
        (
            RS_MODEL_NU
            +
            RS_GAUGE_C
            *
            RS_MODEL_MU
        ),
    )
)


RS_GAUGE_ERROR = (
    np.linalg.norm(
        RS_F_GAUGE_1
        -
        RS_F_GAUGE_0
    )
    /
    max(
        np.linalg.norm(
            RS_F_GAUGE_0
        ),
        np.finfo(float).tiny,
    )
)


# ---------------------------------------------------------------
# 9c. Conservation
# ---------------------------------------------------------------

RS_FEFF_CONSERVATION_ERROR = float(
    abs(
        np.sum(
            RS_F_GAUGE_0
        )
    )
)


# ---------------------------------------------------------------
# 9d. Rank-2 Lie tensor
# ---------------------------------------------------------------

RS_MODEL_A = (
    rs_lie_coefficient_matrix(
        RS_MODEL_MU,
        RS_MODEL_NU,
    )
)


RS_MODEL_A_S = np.linalg.svd(
    RS_MODEL_A,
    compute_uv=False,
)


RS_MODEL_A_REL = (
    RS_MODEL_A_S
    /
    RS_MODEL_A_S[0]
)


RS_MODEL_A_RANK = int(
    np.sum(
        RS_MODEL_A_REL
        >
        1e-12
    )
)


# ================================================================
# 10. Assertions
# ================================================================

assert (
    RS_FORCING_ERROR
    <
    1e-12
)

assert (
    RS_GAUGE_ERROR
    <
    1e-12
)

assert (
    RS_FEFF_CONSERVATION_ERROR
    <
    1e-12
)

assert (
    RS_MODEL_A_RANK
    ==
    2
)


# ================================================================
# 11. Report
# ================================================================

print("=" * 88)
print("NEW RS-TSC STUDY — CELL 4")
print("TSC-CONSTRAINED RANK-2 EFFECTIVE MODEL")
print("=" * 88)


print()
print("Model")
print("-" * 88)

print(
    "F_eff = G_mu + B_chr(G_mu,G_nu)"
)

print(
    f"mu coordinates               : "
    f"{RS_Q}"
)

print(
    f"nu coordinates               : "
    f"{RS_Q}"
)

print(
    "nominal parameter count      : "
    f"{2 * RS_Q}"
)

print(
    "temporal gauge redundancy    : "
    "nu -> nu + c mu"
)

print(
    "intrinsic parameter count    : "
    f"<= {2 * RS_Q - 1}"
)


print()
print("First-Lie structural reachability")
print("-" * 88)

print(
    f"overlapping candidate pairs  : "
    f"{len(RS_FIRST_LIE_META)}"
)

display(
    RS_FIRST_LIE_SUMMARY
)


print()
print(
    "pairwise-only first-depth "
    "order>=4 candidates          : "
    f"{RS_PAIRPAIR_ORDER4PLUS}"
)


print()
print("Exact structural checks")
print("-" * 88)

print(
    f"forcing-matrix equivalence   : "
    f"{RS_FORCING_ERROR:.3e}"
)

print(
    f"gauge invariance error       : "
    f"{RS_GAUGE_ERROR:.3e}"
)

print(
    f"conservation error           : "
    f"{RS_FEFF_CONSERVATION_ERROR:.3e}"
)

print(
    f"Lie coefficient rank         : "
    f"{RS_MODEL_A_RANK}"
)

print(
    "leading relative singular values:"
)

print(
    RS_MODEL_A_REL[
        :8
    ]
)


print()
print("=" * 88)
print("Cell 4 PASSED.")
print("=" * 88)

NEW RS-TSC STUDY — CELL 4
TSC-CONSTRAINED RANK-2 EFFECTIVE MODEL

Model
----------------------------------------------------------------------------------------
F_eff = G_mu + B_chr(G_mu,G_nu)
mu coordinates               : 84
nu coordinates               : 84
nominal parameter count      : 168
temporal gauge redundancy    : nu -> nu + c mu
intrinsic parameter count    : <= 167

First-Lie structural reachability
----------------------------------------------------------------------------------------
overlapping candidate pairs  : 2436


,parent_type,support_order,n_possible_channels
0,pair-pair,3,168
1,pair-triad,3,168
2,pair-triad,4,840
3,triad-triad,4,420
4,triad-triad,5,840



pairwise-only first-depth order>=4 candidates          : 0

Exact structural checks
----------------------------------------------------------------------------------------
forcing-matrix equivalence   : 6.317e-16
gauge invariance error       : 5.357e-16
conservation error           : 2.220e-16
Lie coefficient rank         : 2
leading relative singular values:
[1.00000000e+00 1.00000000e+00 6.57742718e-16 5.26349474e-16
 4.97382674e-16 4.12107625e-16 3.86145616e-16 3.81533044e-16]

Cell 4 PASSED.


Time Resolution Scanning Inference Algorithm

In [13]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# NEW Cell 5:
# direct sparse finite-flow solver for the TSC rank-2 model
#
# INPUT SEEN BY SOLVER
# ---------------------------------------------------------------
#
#   X0, XF, DeltaT
#   admissible interaction library Psi_q
#
# NO:
#   G1 / G2
#   true topology
#   true coefficients
#   BCH oracle
#
#
# SOLVER
# ---------------------------------------------------------------
#
# Stage A:
#   secant Ridge pilot for mu
#   (initialization only)
#
# Stage B:
#   sparse finite-flow inference of mu
#
#       XF ~= Phi_{G_mu}^{DeltaT}(X0)
#
#   using exact endpoint sensitivities + iterative
#   adaptive-LASSO linearization.
#
# Stage C:
#   nonlinear active-set refit of mu.
#
# Stage D:
#   infer temporal coordinate nu from
#
#       endpoint residual ~= S_nu nu
#
#   where S_nu is the FINITE-FLOW temporal sensitivity.
#
# Stage E:
#   joint active-set nonlinear refit of (mu,nu).
#
# ================================================================

import numpy as np
import pandas as pd

from scipy.optimize import least_squares
from sklearn.linear_model import Lasso


# ================================================================
# 0. Numerical helpers
# ================================================================

def rs5_rk4_step(
    rhs,
    y,
    dt,
):

    k1 = rhs(y)

    k2 = rhs(
        y
        +
        0.5 * dt * k1
    )

    k3 = rhs(
        y
        +
        0.5 * dt * k2
    )

    k4 = rhs(
        y
        +
        dt * k3
    )

    return (
        y
        +
        (dt / 6.0)
        *
        (
            k1
            +
            2.0 * k2
            +
            2.0 * k3
            +
            k4
        )
    )


def rs5_n_steps(
    DeltaT,
    dt,
):

    n_steps = int(
        round(
            DeltaT / dt
        )
    )

    assert np.isclose(
        n_steps * dt,
        DeltaT,
        atol=1e-12,
        rtol=0.0,
    )

    return n_steps


# ================================================================
# 1. Adaptive-LASSO path for a generic linearized problem
#
# Solve
#
#       y ~= X beta
#
# after feature normalization.
#
# Adaptive weights are built in normalized-coordinate space.
#
# Returns one beta per lambda.
# ================================================================

def rs5_adaptive_lasso_path(
    X,
    y,
    beta_pilot=None,
    ridge_pilot=1e-8,
    gamma=1.0,
    n_lambda=30,
    lambda_ratio_min=1e-5,
    max_iter=50000,
):

    X = np.asarray(
        X,
        dtype=float,
    )

    y = np.asarray(
        y,
        dtype=float,
    )


    # ------------------------------------------------------------
    # Feature normalization
    #
    # X beta = Xs z
    #
    # with
    #
    #       Xs_q = X_q / scale_q
    #       z_q  = scale_q beta_q
    # ------------------------------------------------------------

    scale = np.sqrt(
        np.mean(
            X ** 2,
            axis=0,
        )
    )

    scale = np.maximum(
        scale,
        1e-14,
    )

    Xs = (
        X
        /
        scale[
            None,
            :
        ]
    )


    # ------------------------------------------------------------
    # Ridge pilot
    # ------------------------------------------------------------

    if beta_pilot is None:

        z_pilot = np.linalg.solve(
            (
                Xs.T @ Xs
                +
                ridge_pilot
                *
                np.eye(
                    X.shape[1]
                )
            ),
            Xs.T @ y,
        )

    else:

        beta_pilot = np.asarray(
            beta_pilot,
            dtype=float,
        )

        z_pilot = (
            scale
            *
            beta_pilot
        )


    pilot_scale = max(
        float(
            np.max(
                np.abs(
                    z_pilot
                )
            )
        ),
        1e-12,
    )

    adaptive_eps = (
        1e-4
        *
        pilot_scale
    )


    adaptive_weight = (
        1.0
        /
        (
            np.abs(
                z_pilot
            )
            +
            adaptive_eps
        )
        **
        gamma
    )


    # ------------------------------------------------------------
    # Transform weighted L1 into ordinary LASSO
    #
    #       u_q = w_q z_q
    #
    #       Xs z = (Xs / w) u
    # ------------------------------------------------------------

    X_lasso = (
        Xs
        /
        adaptive_weight[
            None,
            :
        ]
    )


    n_obs = len(
        y
    )

    lambda_max = (
        np.max(
            np.abs(
                X_lasso.T
                @
                y
            )
        )
        /
        max(
            n_obs,
            1,
        )
    )

    lambda_max = max(
        float(
            lambda_max
        ),
        1e-14,
    )


    lambda_grid = np.geomspace(
        lambda_max,
        lambda_max
        *
        lambda_ratio_min,
        int(
            n_lambda
        ),
    )


    beta_path = []


    for lam in lambda_grid:

        model = Lasso(

            alpha=float(
                lam
            ),

            fit_intercept=False,

            max_iter=int(
                max_iter
            ),

            tol=1e-10,

            selection="cyclic",
        )


        model.fit(
            X_lasso,
            y,
        )


        u = model.coef_

        z = (
            u
            /
            adaptive_weight
        )

        beta = (
            z
            /
            scale
        )


        beta_path.append(
            beta
        )


    return {

        "lambda_grid":
            lambda_grid,

        "beta_path":
            beta_path,

        "scale":
            scale,

        "adaptive_weight":
            adaptive_weight,

        "ridge_pilot_normalized":
            z_pilot,
    }


# ================================================================
# 2. Main solver
# ================================================================

class RSTSCRank2Solver:

    def __init__(
        self,
        DeltaT,
        dt,
    ):

        self.DeltaT = float(
            DeltaT
        )

        self.dt = float(
            dt
        )

        self.N = int(
            N
        )

        self.Q = int(
            RS_Q
        )

        self.n_steps = rs5_n_steps(
            self.DeltaT,
            self.dt,
        )


    # ============================================================
    # 3. Forward integration
    # ============================================================

    def integrate_m0(
        self,
        x0,
        mu,
    ):

        x = np.asarray(
            x0,
            dtype=float,
        ).copy()


        for _ in range(
            self.n_steps
        ):

            x = rs5_rk4_step(

                lambda z:
                    rs_G(
                        z,
                        mu,
                    ),

                x,

                self.dt,
            )


        return x


    def integrate_m1(
        self,
        x0,
        mu,
        nu,
    ):

        x = np.asarray(
            x0,
            dtype=float,
        ).copy()


        for _ in range(
            self.n_steps
        ):

            x = rs5_rk4_step(

                lambda z:
                    rs_Feff_rank2(
                        z,
                        mu,
                        nu,
                    ),

                x,

                self.dt,
            )


        return x


    # ============================================================
    # 4. M0 finite-flow parameter sensitivity
    #
    # For
    #
    #       dx/dt = G_mu(x),
    #
    # endpoint sensitivity
    #
    #       S = dx / dmu
    #
    # satisfies
    #
    #       dS/dt = DG_mu S + Psi(x).
    # ============================================================

    def integrate_m0_sensitivity(
        self,
        x0,
        mu,
    ):

        x0 = np.asarray(
            x0,
            dtype=float,
        )

        S0 = np.zeros(
            (
                self.N,
                self.Q,
            ),
            dtype=float,
        )


        y = np.concatenate(
            [
                x0,
                S0.ravel(),
            ]
        )


        def rhs(
            y_aug,
        ):

            x = y_aug[
                :self.N
            ]

            S = y_aug[
                self.N:
            ].reshape(
                self.N,
                self.Q,
            )


            F = rs_G(
                x,
                mu,
            )

            J = rs_DG(
                x,
                mu,
            )

            Psi = rs_basis_matrix(
                x
            )


            dS = (
                J @ S
                +
                Psi
            )


            return np.concatenate(
                [
                    F,
                    dS.ravel(),
                ]
            )


        for _ in range(
            self.n_steps
        ):

            y = rs5_rk4_step(
                rhs,
                y,
                self.dt,
            )


        xF = y[
            :self.N
        ]

        SF = y[
            self.N:
        ].reshape(
            self.N,
            self.Q,
        )


        return (
            xF,
            SF,
        )


    # ============================================================
    # 5. Temporal finite-flow sensitivity
    #
    # Around M0:
    #
    #       F = G_mu
    #
    # temporal perturbation:
    #
    #       delta F
    #         = H_mu(x) nu
    #
    # where
    #
    #       H_mu = temporal_forcing_matrix.
    #
    # Therefore
    #
    #       dS_nu/dt
    #         = DG_mu S_nu + H_mu.
    # ============================================================

    def integrate_temporal_sensitivity(
        self,
        x0,
        mu,
    ):

        x0 = np.asarray(
            x0,
            dtype=float,
        )


        S0 = np.zeros(
            (
                self.N,
                self.Q,
            ),
            dtype=float,
        )


        y = np.concatenate(
            [
                x0,
                S0.ravel(),
            ]
        )


        def rhs(
            y_aug,
        ):

            x = y_aug[
                :self.N
            ]

            S = y_aug[
                self.N:
            ].reshape(
                self.N,
                self.Q,
            )


            F0 = rs_G(
                x,
                mu,
            )

            J0 = rs_DG(
                x,
                mu,
            )

            H = rs_temporal_forcing_matrix(
                x,
                mu,
            )


            dS = (
                J0 @ S
                +
                H
            )


            return np.concatenate(
                [
                    F0,
                    dS.ravel(),
                ]
            )


        for _ in range(
            self.n_steps
        ):

            y = rs5_rk4_step(
                rhs,
                y,
                self.dt,
            )


        xF0 = y[
            :self.N
        ]

        Snu = y[
            self.N:
        ].reshape(
            self.N,
            self.Q,
        )


        return (
            xF0,
            Snu,
        )


    # ============================================================
    # 6. Dataset normalization
    #
    # Each transition is divided by its observed displacement
    # magnitude.
    #
    # This stops early high-amplitude cycles from dominating the
    # later, more relaxed cycles.
    # ============================================================

    def window_scales(
        self,
        X0,
        XF,
    ):

        scale = np.linalg.norm(
            XF - X0,
            axis=1,
        )

        return np.maximum(
            scale,
            1e-12,
        )


    # ============================================================
    # 7. Endpoint prediction
    # ============================================================

    def predict_m0(
        self,
        X0,
        mu,
    ):

        return np.asarray(
            [
                self.integrate_m0(
                    x0,
                    mu,
                )

                for x0 in X0
            ]
        )


    def predict_m1(
        self,
        X0,
        mu,
        nu,
    ):

        return np.asarray(
            [
                self.integrate_m1(
                    x0,
                    mu,
                    nu,
                )

                for x0 in X0
            ]
        )


    # ============================================================
    # 8. Relative endpoint error
    #
    # Same physical metric used in the earlier forward benchmark:
    #
    #       ||Xpred-XF|| / ||XF-X0||
    # ============================================================

    def endpoint_error(
        self,
        X0,
        XF,
        Xpred,
    ):

        numerator = np.linalg.norm(
            Xpred
            -
            XF
        )

        denominator = np.linalg.norm(
            XF
            -
            X0
        )


        return float(
            numerator
            /
            max(
                denominator,
                np.finfo(float).tiny,
            )
        )


    # ============================================================
    # 9. Secant Ridge initialization for mu
    #
    # IMPORTANT:
    # This is ONLY a numerical starting point.
    #
    # The final estimator remains finite-flow based.
    # ============================================================

    def secant_ridge_mu(
        self,
        X0,
        XF,
        ridge=1e-8,
    ):

        X0 = np.asarray(
            X0,
            dtype=float,
        )

        XF = np.asarray(
            XF,
            dtype=float,
        )


        scales = self.window_scales(
            X0,
            XF,
        )


        A_blocks = []
        y_blocks = []


        for r in range(
            len(X0)
        ):

            Psi = rs_basis_matrix(
                X0[r]
            )

            y = (
                XF[r]
                -
                X0[r]
            ) / self.DeltaT


            A_blocks.append(
                Psi
                /
                scales[r]
            )

            y_blocks.append(
                y
                /
                scales[r]
            )


        A = np.vstack(
            A_blocks
        )

        y = np.concatenate(
            y_blocks
        )


        return np.linalg.solve(
            (
                A.T @ A
                +
                ridge
                *
                np.eye(
                    self.Q
                )
            ),
            A.T @ y,
        )


    # ============================================================
    # 10. Build M0 finite-flow linearization
    #
    # At current mu:
    #
    #       XF
    #         ~= Xpred(mu)
    #            + S_mu (mu_new - mu)
    #
    # therefore
    #
    #       S_mu mu_new
    #         ~= XF - Xpred + S_mu mu.
    #
    # The resulting linearized problem can be solved sparsely.
    # ============================================================

    def build_m0_linearized_problem(
        self,
        X0,
        XF,
        mu,
    ):

        scales = self.window_scales(
            X0,
            XF,
        )


        X_blocks = []
        y_blocks = []

        predictions = []


        for r in range(
            len(X0)
        ):

            xpred, S = (
                self.integrate_m0_sensitivity(
                    X0[r],
                    mu,
                )
            )


            target = (
                XF[r]
                -
                xpred
                +
                S @ mu
            )


            X_blocks.append(
                S
                /
                scales[r]
            )

            y_blocks.append(
                target
                /
                scales[r]
            )

            predictions.append(
                xpred
            )


        return {

            "X":
                np.vstack(
                    X_blocks
                ),

            "y":
                np.concatenate(
                    y_blocks
                ),

            "predictions":
                np.asarray(
                    predictions
                ),
        }


    # ============================================================
    # 11. Sparse nonlinear M0 fit
    #
    # Iterative:
    #
    #   current mu
    #       ->
    #   finite-flow sensitivity
    #       ->
    #   adaptive-LASSO path
    #       ->
    #   choose lambda by FULL nonlinear validation flow
    #
    # No one-SE rule.
    # ============================================================

    def fit_mu_sparse(
        self,
        X0_fit,
        XF_fit,
        X0_val,
        XF_val,
        n_outer=2,
        ridge_init=1e-8,
        ridge_pilot=1e-8,
        gamma=1.0,
        n_lambda=30,
        lambda_ratio_min=1e-5,
    ):

        mu = self.secant_ridge_mu(
            X0_fit,
            XF_fit,
            ridge=ridge_init,
        )


        history = []


        for outer in range(
            int(
                n_outer
            )
        ):

            problem = (
                self.build_m0_linearized_problem(
                    X0_fit,
                    XF_fit,
                    mu,
                )
            )


            path = rs5_adaptive_lasso_path(

                problem[
                    "X"
                ],

                problem[
                    "y"
                ],

                beta_pilot=mu,

                ridge_pilot=ridge_pilot,

                gamma=gamma,

                n_lambda=n_lambda,

                lambda_ratio_min=
                    lambda_ratio_min,
            )


            records = []


            for idx, (
                lam,
                candidate,
            ) in enumerate(
                zip(
                    path[
                        "lambda_grid"
                    ],
                    path[
                        "beta_path"
                    ],
                )
            ):

                pred_val = self.predict_m0(
                    X0_val,
                    candidate,
                )


                val_error = self.endpoint_error(
                    X0_val,
                    XF_val,
                    pred_val,
                )


                active = int(
                    np.count_nonzero(
                        candidate
                    )
                )


                records.append({

                    "outer":
                        outer,

                    "path_index":
                        idx,

                    "lambda":
                        float(
                            lam
                        ),

                    "active":
                        active,

                    "validation_error":
                        float(
                            val_error
                        ),
                })


            val_errors = np.asarray(
                [
                    row[
                        "validation_error"
                    ]
                    for row in records
                ]
            )


            best = int(
                np.argmin(
                    val_errors
                )
            )


            mu_new = (
                path[
                    "beta_path"
                ][best]
                .copy()
            )


            history.extend(
                records
            )


            # Stop if the finite-flow iterate has stabilized.
            denom = max(
                np.linalg.norm(
                    mu
                ),
                1e-12,
            )

            relative_update = (
                np.linalg.norm(
                    mu_new
                    -
                    mu
                )
                /
                denom
            )


            mu = mu_new


            if (
                relative_update
                <
                1e-3
            ):

                break


        return {

            "mu":
                mu,

            "history":
                pd.DataFrame(
                    history
                ),
        }


    # ============================================================
    # 12. Active-set nonlinear refit of mu
    #
    # Once sparse support has been selected, remove the L1 bias
    # and refit only those coordinates using the exact finite flow.
    # ============================================================

    def refit_mu_active(
        self,
        X0_fit,
        XF_fit,
        mu,
        active_rel_tol=1e-8,
        max_nfev=80,
    ):

        mu = np.asarray(
            mu,
            dtype=float,
        )


        scale = max(
            float(
                np.max(
                    np.abs(
                        mu
                    )
                )
            ),
            1e-14,
        )


        active = np.flatnonzero(
            np.abs(
                mu
            )
            >
            active_rel_tol
            *
            scale
        )


        if len(
            active
        ) == 0:

            raise RuntimeError(
                "Sparse M0 fit selected no active coordinates."
            )


        p0 = (
            mu[
                active
            ]
            .copy()
        )


        scales = self.window_scales(
            X0_fit,
            XF_fit,
        )


        def unpack(
            p,
        ):

            beta = np.zeros(
                self.Q,
                dtype=float,
            )

            beta[
                active
            ] = p

            return beta


        def residual(
            p,
        ):

            beta = unpack(
                p
            )


            blocks = []


            for r in range(
                len(X0_fit)
            ):

                pred = self.integrate_m0(
                    X0_fit[r],
                    beta,
                )


                blocks.append(
                    (
                        pred
                        -
                        XF_fit[r]
                    )
                    /
                    scales[r]
                )


            return np.concatenate(
                blocks
            )


        result = least_squares(

            residual,

            p0,

            jac="2-point",

            method="trf",

            x_scale="jac",

            max_nfev=int(
                max_nfev
            ),

            ftol=1e-11,
            xtol=1e-11,
            gtol=1e-11,
        )


        mu_refit = unpack(
            result.x
        )


        return {

            "mu":
                mu_refit,

            "active":
                active,

            "result":
                result,
        }


    # ============================================================
    # 13. Build finite-flow temporal inverse problem
    #
    # Around the fitted native model:
    #
    #       XF - Phi_mu(X0)
    #
    # is explained by
    #
    #       S_nu nu.
    # ============================================================

    def build_temporal_problem(
        self,
        X0,
        XF,
        mu,
    ):

        scales = self.window_scales(
            X0,
            XF,
        )


        X_blocks = []
        y_blocks = []

        predictions = []


        for r in range(
            len(X0)
        ):

            xpred, Snu = (
                self.integrate_temporal_sensitivity(
                    X0[r],
                    mu,
                )
            )


            residual = (
                XF[r]
                -
                xpred
            )


            X_blocks.append(
                Snu
                /
                scales[r]
            )

            y_blocks.append(
                residual
                /
                scales[r]
            )

            predictions.append(
                xpred
            )


        return {

            "X":
                np.vstack(
                    X_blocks
                ),

            "y":
                np.concatenate(
                    y_blocks
                ),

            "predictions":
                np.asarray(
                    predictions
                ),
        }


    # ============================================================
    # 14. Sparse temporal coordinate fit
    #
    # Lambda selection uses the FULL nonlinear M1 validation flow,
    # not merely the linearized temporal residual.
    # ============================================================

    def fit_nu_sparse(
        self,
        X0_fit,
        XF_fit,
        X0_val,
        XF_val,
        mu,
        ridge_pilot=1e-8,
        gamma=1.0,
        n_lambda=36,
        lambda_ratio_min=1e-6,
    ):

        problem = (
            self.build_temporal_problem(
                X0_fit,
                XF_fit,
                mu,
            )
        )


        # Ridge pilot in the actual temporal sensitivity geometry.
        X = problem[
            "X"
        ]

        y = problem[
            "y"
        ]


        nu_pilot = np.linalg.solve(
            (
                X.T @ X
                +
                ridge_pilot
                *
                np.eye(
                    self.Q
                )
            ),
            X.T @ y,
        )


        path = rs5_adaptive_lasso_path(

            X,
            y,

            beta_pilot=nu_pilot,

            ridge_pilot=ridge_pilot,

            gamma=gamma,

            n_lambda=n_lambda,

            lambda_ratio_min=
                lambda_ratio_min,
        )


        records = []


        for idx, (
            lam,
            nu,
        ) in enumerate(
            zip(
                path[
                    "lambda_grid"
                ],
                path[
                    "beta_path"
                ],
            )
        ):

            pred_val = self.predict_m1(
                X0_val,
                mu,
                nu,
            )


            val_error = self.endpoint_error(
                X0_val,
                XF_val,
                pred_val,
            )


            linear_fit_error = (
                np.linalg.norm(
                    X @ nu
                    -
                    y
                )
                /
                max(
                    np.linalg.norm(
                        y
                    ),
                    np.finfo(float).tiny,
                )
            )


            records.append({

                "path_index":
                    idx,

                "lambda":
                    float(
                        lam
                    ),

                "active":
                    int(
                        np.count_nonzero(
                            nu
                        )
                    ),

                "linear_fit_error":
                    float(
                        linear_fit_error
                    ),

                "validation_fullflow_error":
                    float(
                        val_error
                    ),
            })


        val_errors = np.asarray(
            [
                row[
                    "validation_fullflow_error"
                ]
                for row in records
            ]
        )


        best = int(
            np.argmin(
                val_errors
            )
        )


        return {

            "nu":
                path[
                    "beta_path"
                ][best]
                .copy(),

            "pilot":
                nu_pilot,

            "best_index":
                best,

            "best_lambda":
                float(
                    path[
                        "lambda_grid"
                    ][best]
                ),

            "history":
                pd.DataFrame(
                    records
                ),

            "problem":
                problem,
        }


    # ============================================================
    # 15. Joint active-set nonlinear M1 refit
    #
    # The sparse supports are frozen.
    #
    # Only selected mu and nu coefficients are optimized.
    #
    # Numerical differentiation is acceptable here because the
    # selected working set should be small.
    # ============================================================

    def refit_joint_active(
        self,
        X0_fit,
        XF_fit,
        mu,
        nu,
        active_rel_tol=1e-8,
        max_nfev=100,
    ):

        mu = np.asarray(
            mu,
            dtype=float,
        )

        nu = np.asarray(
            nu,
            dtype=float,
        )


        mu_scale = max(
            float(
                np.max(
                    np.abs(
                        mu
                    )
                )
            ),
            1e-14,
        )

        nu_scale = max(
            float(
                np.max(
                    np.abs(
                        nu
                    )
                )
            ),
            1e-14,
        )


        active_mu = np.flatnonzero(
            np.abs(
                mu
            )
            >
            active_rel_tol
            *
            mu_scale
        )


        active_nu = np.flatnonzero(
            np.abs(
                nu
            )
            >
            active_rel_tol
            *
            nu_scale
        )


        if len(
            active_mu
        ) == 0:

            raise RuntimeError(
                "No active mu coordinates."
            )


        p0 = np.concatenate(
            [
                mu[
                    active_mu
                ],

                nu[
                    active_nu
                ],
            ]
        )


        scales = self.window_scales(
            X0_fit,
            XF_fit,
        )


        def unpack(
            p,
        ):

            mu_full = np.zeros(
                self.Q,
                dtype=float,
            )

            nu_full = np.zeros(
                self.Q,
                dtype=float,
            )


            n_mu = len(
                active_mu
            )


            mu_full[
                active_mu
            ] = p[
                :n_mu
            ]

            if len(
                active_nu
            ) > 0:

                nu_full[
                    active_nu
                ] = p[
                    n_mu:
                ]


            return (
                mu_full,
                nu_full,
            )


        def residual(
            p,
        ):

            mu_full, nu_full = (
                unpack(
                    p
                )
            )


            blocks = []


            for r in range(
                len(X0_fit)
            ):

                pred = self.integrate_m1(
                    X0_fit[r],
                    mu_full,
                    nu_full,
                )


                blocks.append(
                    (
                        pred
                        -
                        XF_fit[r]
                    )
                    /
                    scales[r]
                )


            return np.concatenate(
                blocks
            )


        result = least_squares(

            residual,

            p0,

            jac="2-point",

            method="trf",

            x_scale="jac",

            max_nfev=int(
                max_nfev
            ),

            ftol=1e-11,
            xtol=1e-11,
            gtol=1e-11,
        )


        mu_final, nu_final = (
            unpack(
                result.x
            )
        )


        return {

            "mu":
                mu_final,

            "nu":
                nu_final,

            "active_mu":
                active_mu,

            "active_nu":
                active_nu,

            "result":
                result,
        }


# ================================================================
# 16. Instantiate solver
# ================================================================

RS_SOLVER = RSTSCRank2Solver(

    DeltaT=
        RS_CASE2_DIRECT_DATA[
            "DeltaT"
        ],

    dt=
        DT_OBS,
)


# ================================================================
# 17. Smoke test
#
# Only checks dimensions / numerics.
# Does NOT perform inference.
# ================================================================

RS5_TEST_MU = np.zeros(
    RS_Q,
    dtype=float,
)

RS5_TEST_MU[0] = 1.0


RS5_TEST_NU = np.zeros(
    RS_Q,
    dtype=float,
)

RS5_TEST_NU[1] = 0.001


RS5_TEST_X0 = (
    RS_CASE2_DIRECT_DATA[
        "X0_fit"
    ][0]
)


RS5_TEST_M0 = (
    RS_SOLVER.integrate_m0(
        RS5_TEST_X0,
        RS5_TEST_MU,
    )
)


RS5_TEST_M1 = (
    RS_SOLVER.integrate_m1(
        RS5_TEST_X0,
        RS5_TEST_MU,
        RS5_TEST_NU,
    )
)


RS5_TEST_XF_SENS, RS5_TEST_SENS = (
    RS_SOLVER.integrate_m0_sensitivity(
        RS5_TEST_X0,
        RS5_TEST_MU,
    )
)


RS5_TEST_XF_TEMP, RS5_TEST_TEMP_SENS = (
    RS_SOLVER.integrate_temporal_sensitivity(
        RS5_TEST_X0,
        RS5_TEST_MU,
    )
)


assert RS5_TEST_M0.shape == (
    N,
)

assert RS5_TEST_M1.shape == (
    N,
)

assert RS5_TEST_SENS.shape == (
    N,
    RS_Q,
)

assert RS5_TEST_TEMP_SENS.shape == (
    N,
    RS_Q,
)

assert np.all(
    np.isfinite(
        RS5_TEST_TEMP_SENS
    )
)


# ================================================================
# 18. Report
# ================================================================

print("=" * 88)
print("NEW RS-TSC STUDY — CELL 5")
print("DIRECT SPARSE FINITE-FLOW SOLVER")
print("=" * 88)

print(
    f"N                            : "
    f"{RS_SOLVER.N}"
)

print(
    f"admissible coordinates       : "
    f"{RS_SOLVER.Q}"
)

print(
    f"DeltaT                       : "
    f"{RS_SOLVER.DeltaT:.6f}"
)

print(
    f"integration dt               : "
    f"{RS_SOLVER.dt:.6f}"
)

print(
    f"RK4 steps per transition     : "
    f"{RS_SOLVER.n_steps}"
)

print()

print(
    "M0 inference                : "
    "sparse finite-flow"
)

print(
    "temporal inference          : "
    "finite-flow sensitivity + adaptive LASSO"
)

print(
    "final estimator             : "
    "joint active-set finite-flow refit"
)

print()

print(
    f"M0 sensitivity shape         : "
    f"{RS5_TEST_SENS.shape}"
)

print(
    f"temporal sensitivity shape   : "
    f"{RS5_TEST_TEMP_SENS.shape}"
)

print()

print(
    "true topology used           : NO"
)

print(
    "G1/G2 used                   : NO"
)

print(
    "BCH oracle used              : NO"
)

print()
print("=" * 88)
print("Cell 5 PASSED — solver ready.")
print("=" * 88)

NEW RS-TSC STUDY — CELL 5
DIRECT SPARSE FINITE-FLOW SOLVER
N                            : 8
admissible coordinates       : 84
DeltaT                       : 0.030000
integration dt               : 0.000500
RK4 steps per transition     : 60

M0 inference                : sparse finite-flow
temporal inference          : finite-flow sensitivity + adaptive LASSO
final estimator             : joint active-set finite-flow refit

M0 sensitivity shape         : (8, 84)
temporal sensitivity shape   : (8, 84)

true topology used           : NO
G1/G2 used                   : NO
BCH oracle used              : NO

Cell 5 PASSED — solver ready.


In [14]:
# ================================================================
# Stage 6 — N=8 one-time-series slicing
# NEW Cell 6:
# FIRST BLIND TEMPORAL HIGHER-ORDER RECONSTRUCTION
#
# ================================================================
#
# This is the decisive proof-of-concept experiment.
#
# INPUT:
#
#   12 fit transitions
#    3 validation transitions
#    3 untouched external-test transitions
#
# Solver knows only:
#
#   - X0
#   - XF
#   - DeltaT
#   - admissible pair / triad interaction functions
#
# Solver does NOT know:
#
#   - G1
#   - G2
#   - true pair topology
#   - true triad topology
#   - microscopic coefficients
#   - BCH coefficients
#
#
# Pipeline:
#
#   1. sparse finite-flow inference of mu
#   2. active-set nonlinear refit of mu
#   3. finite-flow temporal sensitivity
#   4. sparse inference of nu
#   5. active-set joint nonlinear refit
#
# ================================================================

import time
import numpy as np
import pandas as pd


# ================================================================
# 0. Observable-only data
# ================================================================

X0_fit = np.asarray(
    RS_CASE2_DIRECT_DATA[
        "X0_fit"
    ],
    dtype=float,
)

XF_fit = np.asarray(
    RS_CASE2_DIRECT_DATA[
        "XF_fit"
    ],
    dtype=float,
)


X0_val = np.asarray(
    RS_CASE2_DIRECT_DATA[
        "X0_validation"
    ],
    dtype=float,
)

XF_val = np.asarray(
    RS_CASE2_DIRECT_DATA[
        "XF_validation"
    ],
    dtype=float,
)


X0_test = np.asarray(
    RS_CASE2_DIRECT_DATA[
        "X0_test"
    ],
    dtype=float,
)

XF_test = np.asarray(
    RS_CASE2_DIRECT_DATA[
        "XF_test"
    ],
    dtype=float,
)


assert X0_fit.shape == (
    12,
    N,
)

assert X0_val.shape == (
    3,
    N,
)

assert X0_test.shape == (
    3,
    N,
)


print("=" * 92)
print("NEW RS-TSC STUDY — CELL 6")
print("BLIND CASE-2 TEMPORAL HIGHER-ORDER RECONSTRUCTION")
print("=" * 92)

print()
print("Data")
print("-" * 92)

print(
    f"fit transitions             : "
    f"{len(X0_fit)}"
)

print(
    f"validation transitions      : "
    f"{len(X0_val)}"
)

print(
    f"external test transitions   : "
    f"{len(X0_test)}"
)

print(
    f"DeltaT                      : "
    f"{RS_SOLVER.DeltaT:.6f}"
)

print()

print(
    "true topology visible       : NO"
)

print(
    "G1/G2 visible               : NO"
)

print(
    "BCH oracle visible          : NO"
)


# ================================================================
# 1. Utility: evaluate one model on all three splits
# ================================================================

def rs6_evaluate(
    label,
    mu,
    nu=None,
):

    if nu is None:

        pred_fit = (
            RS_SOLVER.predict_m0(
                X0_fit,
                mu,
            )
        )

        pred_val = (
            RS_SOLVER.predict_m0(
                X0_val,
                mu,
            )
        )

        pred_test = (
            RS_SOLVER.predict_m0(
                X0_test,
                mu,
            )
        )

    else:

        pred_fit = (
            RS_SOLVER.predict_m1(
                X0_fit,
                mu,
                nu,
            )
        )

        pred_val = (
            RS_SOLVER.predict_m1(
                X0_val,
                mu,
                nu,
            )
        )

        pred_test = (
            RS_SOLVER.predict_m1(
                X0_test,
                mu,
                nu,
            )
        )


    return {

        "stage":
            label,

        "fit_error":
            RS_SOLVER.endpoint_error(
                X0_fit,
                XF_fit,
                pred_fit,
            ),

        "validation_error":
            RS_SOLVER.endpoint_error(
                X0_val,
                XF_val,
                pred_val,
            ),

        "external_test_error":
            RS_SOLVER.endpoint_error(
                X0_test,
                XF_test,
                pred_test,
            ),

        "pred_fit":
            pred_fit,

        "pred_val":
            pred_val,

        "pred_test":
            pred_test,
    }


# ================================================================
# 2. Stage A/B:
#    sparse finite-flow inference of native coordinate mu
# ================================================================

print()
print("-" * 92)
print("A. SPARSE FINITE-FLOW INFERENCE OF mu")
print("-" * 92)


RS6_T0 = time.perf_counter()


RS6_MU_SPARSE_RESULT = (
    RS_SOLVER.fit_mu_sparse(

        X0_fit,
        XF_fit,

        X0_val,
        XF_val,

        n_outer=2,

        ridge_init=1e-8,

        ridge_pilot=1e-8,

        gamma=1.0,

        n_lambda=32,

        lambda_ratio_min=1e-5,
    )
)


RS6_MU_SPARSE = (
    RS6_MU_SPARSE_RESULT[
        "mu"
    ]
    .copy()
)


RS6_T_MU_SPARSE = (
    time.perf_counter()
    -
    RS6_T0
)


RS6_ACTIVE_MU_SPARSE = (
    np.flatnonzero(
        RS6_MU_SPARSE
        !=
        0.0
    )
)


print(
    f"selected mu coordinates      : "
    f"{len(RS6_ACTIVE_MU_SPARSE)} / {RS_Q}"
)

print(
    f"runtime                      : "
    f"{RS6_T_MU_SPARSE:.2f} s"
)


# Best validation point from each outer iteration
RS6_MU_PATH = (
    RS6_MU_SPARSE_RESULT[
        "history"
    ]
)


RS6_MU_BEST_BY_OUTER = (

    RS6_MU_PATH.loc[
        RS6_MU_PATH
        .groupby(
            "outer"
        )[
            "validation_error"
        ]
        .idxmin()
    ]

    .sort_values(
        "outer"
    )

    .reset_index(
        drop=True
    )
)


display(
    RS6_MU_BEST_BY_OUTER
)


# ================================================================
# 3. Stage C:
#    remove L1 bias by nonlinear active-set refit
# ================================================================

print()
print("-" * 92)
print("B. ACTIVE-SET NONLINEAR REFIT OF mu")
print("-" * 92)


RS6_T1 = time.perf_counter()


RS6_MU_REFIT_RESULT = (
    RS_SOLVER.refit_mu_active(

        X0_fit,
        XF_fit,

        RS6_MU_SPARSE,

        active_rel_tol=1e-8,

        max_nfev=80,
    )
)


RS6_MU = (
    RS6_MU_REFIT_RESULT[
        "mu"
    ]
    .copy()
)


RS6_ACTIVE_MU = (
    RS6_MU_REFIT_RESULT[
        "active"
    ]
)


RS6_T_MU_REFIT = (
    time.perf_counter()
    -
    RS6_T1
)


print(
    f"active mu coordinates        : "
    f"{len(RS6_ACTIVE_MU)}"
)

print(
    f"least-squares success        : "
    f"{RS6_MU_REFIT_RESULT['result'].success}"
)

print(
    f"function evaluations         : "
    f"{RS6_MU_REFIT_RESULT['result'].nfev}"
)

print(
    f"runtime                      : "
    f"{RS6_T_MU_REFIT:.2f} s"
)


# ================================================================
# 4. Evaluate native-only model
# ================================================================

RS6_EVAL_M0_SPARSE = (
    rs6_evaluate(
        "M0_sparse",
        RS6_MU_SPARSE,
        None,
    )
)


RS6_EVAL_M0 = (
    rs6_evaluate(
        "M0_native_refit",
        RS6_MU,
        None,
    )
)


# ================================================================
# 5. Stage D:
#    infer temporal Lie coordinate nu
# ================================================================

print()
print("-" * 92)
print("C. FINITE-FLOW TEMPORAL INFERENCE OF nu")
print("-" * 92)


RS6_T2 = time.perf_counter()


RS6_NU_RESULT = (
    RS_SOLVER.fit_nu_sparse(

        X0_fit,
        XF_fit,

        X0_val,
        XF_val,

        RS6_MU,

        ridge_pilot=1e-8,

        gamma=1.0,

        n_lambda=40,

        lambda_ratio_min=1e-6,
    )
)


RS6_NU = (
    RS6_NU_RESULT[
        "nu"
    ]
    .copy()
)


RS6_T_NU = (
    time.perf_counter()
    -
    RS6_T2
)


RS6_ACTIVE_NU = (
    np.flatnonzero(
        RS6_NU
        !=
        0.0
    )
)


print(
    f"best lambda                   : "
    f"{RS6_NU_RESULT['best_lambda']:.6e}"
)

print(
    f"selected nu coordinates       : "
    f"{len(RS6_ACTIVE_NU)} / {RS_Q}"
)

print(
    f"runtime                       : "
    f"{RS6_T_NU:.2f} s"
)


RS6_NU_PATH = (
    RS6_NU_RESULT[
        "history"
    ]
)


RS6_NU_BEST_ROW = (
    RS6_NU_PATH.iloc[
        RS6_NU_RESULT[
            "best_index"
        ]
    ]
)


print()
print("selected temporal path point:")

display(
    RS6_NU_BEST_ROW
    .to_frame()
    .T
)


# ================================================================
# 6. Evaluate M1 before joint refit
# ================================================================

RS6_EVAL_M1_PRE = (
    rs6_evaluate(
        "M1_temporal_sparse",
        RS6_MU,
        RS6_NU,
    )
)


# ================================================================
# 7. Stage E:
#    joint nonlinear active-set refit
# ================================================================

print()
print("-" * 92)
print("D. JOINT ACTIVE-SET FINITE-FLOW REFIT")
print("-" * 92)


RS6_T3 = time.perf_counter()


RS6_JOINT_RESULT = (
    RS_SOLVER.refit_joint_active(

        X0_fit,
        XF_fit,

        RS6_MU,
        RS6_NU,

        active_rel_tol=1e-8,

        max_nfev=80,
    )
)


RS6_MU_FINAL = (
    RS6_JOINT_RESULT[
        "mu"
    ]
    .copy()
)


RS6_NU_FINAL = (
    RS6_JOINT_RESULT[
        "nu"
    ]
    .copy()
)


RS6_ACTIVE_MU_FINAL = (
    RS6_JOINT_RESULT[
        "active_mu"
    ]
)


RS6_ACTIVE_NU_FINAL = (
    RS6_JOINT_RESULT[
        "active_nu"
    ]
)


RS6_T_JOINT = (
    time.perf_counter()
    -
    RS6_T3
)


print(
    f"active mu coordinates        : "
    f"{len(RS6_ACTIVE_MU_FINAL)}"
)

print(
    f"active nu coordinates        : "
    f"{len(RS6_ACTIVE_NU_FINAL)}"
)

print(
    f"joint parameter count        : "
    f"{len(RS6_ACTIVE_MU_FINAL) + len(RS6_ACTIVE_NU_FINAL)}"
)

print(
    f"least-squares success        : "
    f"{RS6_JOINT_RESULT['result'].success}"
)

print(
    f"function evaluations         : "
    f"{RS6_JOINT_RESULT['result'].nfev}"
)

print(
    f"runtime                      : "
    f"{RS6_T_JOINT:.2f} s"
)


# ================================================================
# 8. Final evaluation
# ================================================================

RS6_EVAL_FINAL = (
    rs6_evaluate(
        "M1_joint_refit",
        RS6_MU_FINAL,
        RS6_NU_FINAL,
    )
)


RS6_EVALUATIONS = [

    RS6_EVAL_M0_SPARSE,
    RS6_EVAL_M0,
    RS6_EVAL_M1_PRE,
    RS6_EVAL_FINAL,
]


RS6_ERROR_TABLE = pd.DataFrame(
    [
        {

            "stage":
                row[
                    "stage"
                ],

            "fit_error":
                row[
                    "fit_error"
                ],

            "validation_error":
                row[
                    "validation_error"
                ],

            "external_test_error":
                row[
                    "external_test_error"
                ],
        }

        for row in
        RS6_EVALUATIONS
    ]
)


print()
print("-" * 92)
print("E. FINITE-FLOW PREDICTION ERRORS")
print("-" * 92)


display(
    RS6_ERROR_TABLE.style.format({

        "fit_error":
            "{:.6e}",

        "validation_error":
            "{:.6e}",

        "external_test_error":
            "{:.6e}",
    })
)


# ================================================================
# 9. Center-window errors
#
# These are especially important because the local-C estimator
# targets the center protocol at offset = 0.
# ================================================================

def rs6_center_error(
    role,
    mu,
    nu=None,
):

    role_mask = (
        RS_CASE2_META[
            "role"
        ].to_numpy()
        ==
        role
    )

    mask = (
        role_mask
        &
        RS_CASE2_CENTER_MASK
    )


    X0 = (
        RS_CASE2_X0[
            mask
        ]
    )

    XF = (
        RS_CASE2_XF[
            mask
        ]
    )


    if nu is None:

        pred = (
            RS_SOLVER.predict_m0(
                X0,
                mu,
            )
        )

    else:

        pred = (
            RS_SOLVER.predict_m1(
                X0,
                mu,
                nu,
            )
        )


    return (
        RS_SOLVER.endpoint_error(
            X0,
            XF,
            pred,
        )
    )


RS6_CENTER_ROWS = []


for (
    label,
    mu,
    nu,
) in [

    (
        "M0_sparse",
        RS6_MU_SPARSE,
        None,
    ),

    (
        "M0_native_refit",
        RS6_MU,
        None,
    ),

    (
        "M1_temporal_sparse",
        RS6_MU,
        RS6_NU,
    ),

    (
        "M1_joint_refit",
        RS6_MU_FINAL,
        RS6_NU_FINAL,
    ),
]:

    RS6_CENTER_ROWS.append({

        "stage":
            label,

        "fit_center_error":
            rs6_center_error(
                "fit",
                mu,
                nu,
            ),

        "validation_center_error":
            rs6_center_error(
                "validation",
                mu,
                nu,
            ),

        "external_center_error":
            rs6_center_error(
                "test",
                mu,
                nu,
            ),
    })


RS6_CENTER_TABLE = pd.DataFrame(
    RS6_CENTER_ROWS
)


print()
print("-" * 92)
print("F. CENTER-PROTOCOL ERRORS")
print("-" * 92)


display(
    RS6_CENTER_TABLE.style.format({

        "fit_center_error":
            "{:.6e}",

        "validation_center_error":
            "{:.6e}",

        "external_center_error":
            "{:.6e}",
    })
)


# ================================================================
# 10. Inferred coordinate tables
#
# Still NO oracle comparison.
# ================================================================

def rs6_coordinate_table(
    beta,
    coordinate_name,
):

    beta = np.asarray(
        beta,
        dtype=float,
    )


    active = np.flatnonzero(
        beta != 0.0
    )


    if len(
        active
    ) == 0:

        return pd.DataFrame(
            columns=[
                "coordinate",
                "q",
                "kind",
                "support",
                "coefficient",
                "abs_coefficient",
            ]
        )


    active = active[
        np.argsort(
            -np.abs(
                beta[
                    active
                ]
            )
        )
    ]


    rows = []


    for q in active:

        rows.append({

            "coordinate":
                coordinate_name,

            "q":
                int(
                    q
                ),

            "kind":
                RS_MICRO_META.loc[
                    q,
                    "kind"
                ],

            "support":
                RS_MICRO_META.loc[
                    q,
                    "support"
                ],

            "coefficient":
                float(
                    beta[q]
                ),

            "abs_coefficient":
                float(
                    abs(
                        beta[q]
                    )
                ),
        })


    return pd.DataFrame(
        rows
    )


RS6_MU_TABLE = (
    rs6_coordinate_table(
        RS6_MU_FINAL,
        "mu",
    )
)


RS6_NU_TABLE = (
    rs6_coordinate_table(
        RS6_NU_FINAL,
        "nu",
    )
)


print()
print("-" * 92)
print("G. INFERRED COORDINATES — NO ORACLE")
print("-" * 92)

print()
print("mu:")

display(
    RS6_MU_TABLE.head(
        30
    )
)


print()
print("nu:")

display(
    RS6_NU_TABLE.head(
        30
    )
)


# ================================================================
# 11. Inferred temporal Lie sector
#
# A = mu ^ nu
#
# Rank <=2 is structural, not fitted independently.
# ================================================================

RS6_A_FINAL = (
    rs_lie_coefficient_matrix(
        RS6_MU_FINAL,
        RS6_NU_FINAL,
    )
)


RS6_A_S = np.linalg.svd(
    RS6_A_FINAL,
    compute_uv=False,
)


if (
    RS6_A_S[0]
    >
    np.finfo(float).tiny
):

    RS6_A_REL = (
        RS6_A_S
        /
        RS6_A_S[0]
    )

else:

    RS6_A_REL = np.zeros_like(
        RS6_A_S
    )


RS6_A_RANK = int(
    np.sum(
        RS6_A_REL
        >
        1e-10
    )
)


print()
print("-" * 92)
print("H. INFERRED TEMPORAL LIE SECTOR")
print("-" * 92)

print(
    f"Lie matrix numerical rank    : "
    f"{RS6_A_RANK}"
)

print(
    "leading relative singular values:"
)

print(
    RS6_A_REL[
        :8
    ]
)


# ================================================================
# 12. Size of inferred temporal correction
#
# Evaluate only on observed center states.
#
# No oracle required.
# ================================================================

RS6_CENTER_STATES = (
    RS_CASE2_X0[
        RS_CASE2_CENTER_MASK
    ]
)


RS6_TEMPORAL_RATIO = []


for x in RS6_CENTER_STATES:

    native = rs_G(
        x,
        RS6_MU_FINAL,
    )

    temporal = (
        rs_temporal_bracket(
            x,
            RS6_MU_FINAL,
            RS6_NU_FINAL,
        )
    )


    ratio = (
        np.linalg.norm(
            temporal
        )
        /
        max(
            np.linalg.norm(
                native
            ),
            np.finfo(float).tiny,
        )
    )


    RS6_TEMPORAL_RATIO.append(
        ratio
    )


RS6_TEMPORAL_RATIO = np.asarray(
    RS6_TEMPORAL_RATIO
)


print()
print("-" * 92)
print("I. INFERRED TEMPORAL-CORRECTION SIZE")
print("-" * 92)

print(
    f"mean ||F_temp||/||F_native|| : "
    f"{np.mean(RS6_TEMPORAL_RATIO):.6e}"
)

print(
    f"min                          : "
    f"{np.min(RS6_TEMPORAL_RATIO):.6e}"
)

print(
    f"max                          : "
    f"{np.max(RS6_TEMPORAL_RATIO):.6e}"
)


# ================================================================
# 13. Main blind decision
# ================================================================

RS6_M0_TEST_ERROR = float(
    RS6_EVAL_M0[
        "external_test_error"
    ]
)


RS6_M1_TEST_ERROR = float(
    RS6_EVAL_FINAL[
        "external_test_error"
    ]
)


RS6_M0_CENTER_TEST = float(
    RS6_CENTER_TABLE.loc[
        RS6_CENTER_TABLE[
            "stage"
        ]
        ==
        "M0_native_refit",
        "external_center_error",
    ].iloc[0]
)


RS6_M1_CENTER_TEST = float(
    RS6_CENTER_TABLE.loc[
        RS6_CENTER_TABLE[
            "stage"
        ]
        ==
        "M1_joint_refit",
        "external_center_error",
    ].iloc[0]
)


RS6_TEST_IMPROVEMENT = (
    RS6_M0_TEST_ERROR
    /
    max(
        RS6_M1_TEST_ERROR,
        1e-30,
    )
)


RS6_CENTER_IMPROVEMENT = (
    RS6_M0_CENTER_TEST
    /
    max(
        RS6_M1_CENTER_TEST,
        1e-30,
    )
)


RS6_TOTAL_RUNTIME = (
    RS6_T_MU_SPARSE
    +
    RS6_T_MU_REFIT
    +
    RS6_T_NU
    +
    RS6_T_JOINT
)


print()
print("=" * 92)
print("J. BLIND RECONSTRUCTION SUMMARY")
print("=" * 92)

print(
    f"M0 external-test error       : "
    f"{RS6_M0_TEST_ERROR:.6e}"
)

print(
    f"M1 external-test error       : "
    f"{RS6_M1_TEST_ERROR:.6e}"
)

print(
    f"test improvement factor      : "
    f"{RS6_TEST_IMPROVEMENT:.3f}"
)

print()

print(
    f"M0 external CENTER error     : "
    f"{RS6_M0_CENTER_TEST:.6e}"
)

print(
    f"M1 external CENTER error     : "
    f"{RS6_M1_CENTER_TEST:.6e}"
)

print(
    f"center improvement factor    : "
    f"{RS6_CENTER_IMPROVEMENT:.3f}"
)

print()

print(
    f"final active mu              : "
    f"{len(RS6_ACTIVE_MU_FINAL)}"
)

print(
    f"final active nu              : "
    f"{len(RS6_ACTIVE_NU_FINAL)}"
)

print(
    f"total active coordinates     : "
    f"{len(RS6_ACTIVE_MU_FINAL) + len(RS6_ACTIVE_NU_FINAL)}"
)

print()

print(
    f"total runtime                : "
    f"{RS6_TOTAL_RUNTIME:.2f} s"
)

print()
print(
    "No synthetic oracle has been used in Cell 6."
)


# ================================================================
# 14. Save blind results
# ================================================================

RS6_ERROR_TABLE.to_csv(
    OUTPUT_DIR
    /
    "rs_new_cell6_blind_endpoint_errors.csv",
    index=False,
)


RS6_CENTER_TABLE.to_csv(
    OUTPUT_DIR
    /
    "rs_new_cell6_blind_center_errors.csv",
    index=False,
)


RS6_MU_PATH.to_csv(
    OUTPUT_DIR
    /
    "rs_new_cell6_mu_lambda_path.csv",
    index=False,
)


RS6_NU_PATH.to_csv(
    OUTPUT_DIR
    /
    "rs_new_cell6_nu_lambda_path.csv",
    index=False,
)


RS6_MU_TABLE.to_csv(
    OUTPUT_DIR
    /
    "rs_new_cell6_inferred_mu.csv",
    index=False,
)


RS6_NU_TABLE.to_csv(
    OUTPUT_DIR
    /
    "rs_new_cell6_inferred_nu.csv",
    index=False,
)


np.savez_compressed(

    OUTPUT_DIR
    /
    "rs_new_cell6_blind_reconstruction.npz",

    mu_sparse=
        RS6_MU_SPARSE,

    mu_native_refit=
        RS6_MU,

    nu_sparse=
        RS6_NU,

    mu_final=
        RS6_MU_FINAL,

    nu_final=
        RS6_NU_FINAL,

    lie_matrix=
        RS6_A_FINAL,

    temporal_ratio=
        RS6_TEMPORAL_RATIO,
)


print()
print("=" * 92)
print("Cell 6 complete.")
print("=" * 92)

NEW RS-TSC STUDY — CELL 6
BLIND CASE-2 TEMPORAL HIGHER-ORDER RECONSTRUCTION

Data
--------------------------------------------------------------------------------------------
fit transitions             : 12
validation transitions      : 3
external test transitions   : 3
DeltaT                      : 0.030000

true topology visible       : NO
G1/G2 visible               : NO
BCH oracle visible          : NO

--------------------------------------------------------------------------------------------
A. SPARSE FINITE-FLOW INFERENCE OF mu
--------------------------------------------------------------------------------------------


C:\Users\liu.xuanc\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.963e-03, tolerance: 1.788e-08
  model = cd_fast.enet_coordinate_descent(
C:\Users\liu.xuanc\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.805e-07, tolerance: 1.788e-08
  model = cd_fast.enet_coordinate_descent(
C:\Users\liu.xuanc\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider incre

selected mu coordinates      : 9 / 84
runtime                      : 9.47 s


,outer,path_index,lambda,active,validation_error
0,0,25,0.016580,10,0.178274
1,1,26,0.000003,9,0.134730



--------------------------------------------------------------------------------------------
B. ACTIVE-SET NONLINEAR REFIT OF mu
--------------------------------------------------------------------------------------------
active mu coordinates        : 9
least-squares success        : True
function evaluations         : 4
runtime                      : 9.37 s

--------------------------------------------------------------------------------------------
C. FINITE-FLOW TEMPORAL INFERENCE OF nu
--------------------------------------------------------------------------------------------


...


--------------------------------------------------------------------------------------------
G. INFERRED COORDINATES — NO ORACLE
--------------------------------------------------------------------------------------------

mu:


,coordinate,q,kind,support,coefficient,abs_coefficient
0,mu,60,triad,"(2, 5, 8)",-3.728433,3.728433
1,mu,63,triad,"(2, 7, 8)",-2.874576,2.874576
2,mu,81,triad,"(5, 6, 8)",-2.869806,2.869806
3,mu,13,pair,"(3, 4)",2.172203,2.172203
4,mu,50,triad,"(2, 3, 5)",-2.165451,2.165451
5,mu,3,pair,"(1, 5)",-1.138455,1.138455
6,mu,53,triad,"(2, 3, 8)",0.567828,0.567828
7,mu,12,pair,"(2, 8)",-0.045912,0.045912
8,mu,26,pair,"(6, 8)",0.012083,0.012083



nu:


,coordinate,q,kind,support,coefficient,abs_coefficient
0,nu,83,triad,"(6, 7, 8)",28.764084,28.764084
1,nu,80,triad,"(5, 6, 7)",-12.873141,12.873141
2,nu,46,triad,"(1, 6, 7)",-11.247172,11.247172
3,nu,44,triad,"(1, 5, 7)",-8.678221,8.678221
4,nu,48,triad,"(1, 7, 8)",7.781472,7.781472
5,nu,10,pair,"(2, 6)",7.716516,7.716516
6,nu,71,triad,"(3, 6, 7)",-6.974339,6.974339
7,nu,5,pair,"(1, 7)",-6.732313,6.732313
8,nu,59,triad,"(2, 5, 7)",5.570545,5.570545
9,nu,63,triad,"(2, 7, 8)",5.442150,5.442150



--------------------------------------------------------------------------------------------
H. INFERRED TEMPORAL LIE SECTOR
--------------------------------------------------------------------------------------------
Lie matrix numerical rank    : 2
leading relative singular values:
[1.00000000e+00 1.00000000e+00 3.17609751e-16 3.16858169e-16
 2.77406677e-16 2.57039930e-16 2.37496208e-16 2.12653991e-16]

--------------------------------------------------------------------------------------------
I. INFERRED TEMPORAL-CORRECTION SIZE
--------------------------------------------------------------------------------------------
mean ||F_temp||/||F_native|| : 4.567151e-01
min                          : 4.263370e-01
max                          : 4.968325e-01

J. BLIND RECONSTRUCTION SUMMARY
M0 external-test error       : 1.920655e-01
M1 external-test error       : 2.304927e-01
test improvement factor      : 0.833

M0 external CENTER error     : 1.907749e-01
M1 external CENTER error     : 2